# Sudoku-06 : Résolution par CSP Académique (Rust)

**Niveau** : Programmation par Contraintes | **Durée** : ~25 min | **Prérequis** : Sudoku-00 Environment

## Navigation

| << Précédent | [Index](README.md) | Suivant >> |
|-------------|---------------------|-----------|
| [Sudoku-05-PSO-Rust](Sudoku-05-PSO-Rust.ipynb) | | [Sudoku-07-Norvig-Rust](Sudoku-07-Norvig-Rust.ipynb) |

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Formaliser** le Sudoku comme un CSP (variables, domaines, contraintes)
2. **Implementer** les algorithmes de référence AIMA : AC-3, Forward Checking, MAC
3. **Appliquer** les heuristiques MRV et LCV pour optimiser la recherche
4. **Comparer** expérimentalement les différentes stratégies de résolution

***

Ce notebook présente la résolution de Sudoku selon l'approche académique décrite dans **"Artificial Intelligence: A Modern Approach"** (Russell & Norvig, Chapitre 6). Tout le code est écrit avec la seule bibliothèque standard de Rust : aucune crate, aucune dépendance externe.

## Introduction

Contrairement aux bibliothèques industrielles (OR-Tools, Choco) ou aux métaheuristiques (GA, SA, PSO), l'approche AIMA :
- **Est pédagogique** : chaque composant est transparent et compréhensible
- **Est modulaire** : on peut combiner différentes heuristiques et propagations
- **Sert de référence** : c'est le standard académique pour comparer les algorithmes

### Formalisation CSP du Sudoku

| Composant | Description | Taille |
|-----------|-------------|-------|
| **Variables** | $X_{i,j}$ pour chaque cellule (i, j) | 81 |
| **Domaines** | $D_{i,j} \subseteq \{1, 2, \ldots, 9\}$ | 9 valeurs max |
| **Contraintes** | AllDifferent par ligne, colonne, bloc 3x3 | 27 contraintes |

### Algorithmes couverts

| Algorithme | Type | Complexité | Puissance |
|------------|------|------------|----------|
| **Backtracking simple** | Recherche | $O(d^n)$ | Faible |
| **Backtracking + MRV** | Heuristique | Variable | Moderee |
| **Forward Checking** | Propagation 1-niveau | $O(n \cdot d^2)$ | Moderee |
| **AC-3** | Arc-consistance | $O(e \cdot d^3)$ | Forte |
| **MAC** | AC-3 + Backtracking | Variable | Très forte |


In [2]:
// Imports
// Aucune crate : la bibliothèque standard de Rust suffit a tout le carnet.
// Le port Python importe numpy, time, copy, typing, collections et enum ;
// l'import numpy est mort (aucune occurrence de `np.` dans le code).

println!("Libraries importees avec succes.");

Libraries importees avec succes.


Configuration du chemin vers les fichiers de puzzles.

La cellule ci-dessous ancre le carnet dans le système de fichiers : `std::env::current_dir()` désigne le répertoire de travail du kernel — le dossier `Sudoku/` quand le carnet est lancé depuis sa série — et `PUZZLES_DIR` pointe vers `Puzzles/`, le dossier qui porte les corpus partagés de la série (grilles faciles et benchmark difficile). Le garde `if PUZZLES_DIR.exists()` affiche un avertissement explicite dès la configuration plutôt qu'une erreur `std::io::Error` au milieu d'un benchmark : un chemin cassé se voit immédiatement, pas après dix minutes de résolution. Le préfixe `<repo>` de la sortie est la convention d'affichage de la série — chemin relatif au dépôt, jamais un chemin machine absolu.

In [3]:
// Configuration du chemin vers les puzzles
use std::path::{Path, PathBuf};

/// Remonte jusqu'au repertoire racine du depot (marqueur .git).
fn racine_depot(depart: &Path) -> PathBuf {
    let mut candidat = Some(depart);
    while let Some(c) = candidat {
        if c.join(".git").exists() {
            return c.to_path_buf();
        }
        candidat = c.parent();
    }
    depart.to_path_buf()
}

let notebook_dir = std::env::current_dir().unwrap_or_else(|_| PathBuf::from("."));
let puzzles_dir = notebook_dir.join("Puzzles");

// Chemin relatif au depot : la sortie commitee ne doit jamais exposer le
// chemin machine d'execution. Convention d'affichage de la serie : prefixe
// `<repo>` (cf. cellule 2).
let racine = racine_depot(&notebook_dir);
let chemin_affiche = match puzzles_dir.strip_prefix(&racine) {
    Ok( relatif) => format!("<repo>{}", relatif.display()),
    Err(_) => format!("<repo>{}", puzzles_dir.file_name().map(|n| n.to_string_lossy().to_string()).unwrap_or_default()),
};

if puzzles_dir.exists() {
    println!("Dossier Puzzles: {}", chemin_affiche);
} else {
    println!("ATTENTION: Dossier Puzzles non trouve ({})", chemin_affiche);
};

Dossier Puzzles: <repo>MyIA.AI.Notebooks/Sudoku/Puzzles


## 1. Classe SudokuGrid

La première brique est délibérément simple : une grille 9×9 en tableau `[[u8; 9]; 9]`, avec 0 pour les cases vides. Autour de cette structure, la structure regroupe tout ce dont les solveurs de la série auront besoin : lecture d'une chaîne de 81 caractères (`from_string`, où le point `.` est accepté comme synonyme de 0), copie indépendante (`clone`, indispensable au backtracking qui doit pouvoir revenir en arrière sans corrompre l'état initial), validation à trois niveaux (`is_valid` : lignes, colonnes, blocs), test de complétude (`is_complete`) et affichage lisible avec séparateurs de blocs (`Display`).

Notons ce que la structure ne sait **pas** faire : elle ne connaît rien aux CSP — ni domaines, ni contraintes, ni propagation. C'est un conteneur neutre ; la fonction `build_csp` de la section 3 chargera de le traduire en instance CSP. Cette séparation des rôles est la même que celle du jumeau C# de la série (Sudoku-06-C#) : la grille est la donnée du problème, le CSP en est le modèle algorithmique, et les solveurs qui suivent ne travaillent que sur le second.

In [4]:
// Classe SudokuGrid
use std::fmt;
use std::path::Path;

/// Representation d'une grille de Sudoku 9x9.
#[derive(Clone)]
struct SudokuGrid {
    cells: [[u8; 9]; 9],
}

impl SudokuGrid {
    fn new() -> Self {
        SudokuGrid { cells: [[0u8; 9]; 9] }
    }

    /// Lecture d'une chaine de 81 caracteres ; '.' est accepte comme synonyme de 0.
    /// Une longueur != 81 est un echec explicite, jamais une grille vide.
    fn from_string(s: &str) -> Result<Self, String> {
        let nettoye: String = s
            .chars()
            .map(|c| if c == '.' { '0' } else { c })
            .filter(|c| *c != ' ' && *c != '\n' && *c != '\r')
            .collect();
        if nettoye.chars().count() != 81 {
            return Err("La chaine doit avoir 81 caracteres".to_string());
        }
        let mut grille = SudokuGrid::new();
        for (i, c) in nettoye.chars().enumerate() {
            let v: u8 = c
                .to_digit(10)
                .ok_or_else(|| "La chaine doit contenir uniquement des chiffres".to_string())? as u8;
            grille.cells[i / 9][i % 9] = v;
        }
        Ok(grille)
    }

    fn clone(&self) -> Self {
        SudokuGrid { cells: self.cells }
    }

    fn to_string(&self) -> String {
        let mut s = String::new();
        for r in 0..9 {
            for c in 0..9 {
                s.push_str(&self.cells[r][c].to_string());
            }
        }
        s
    }

    /// Verifie si la grille est valide (sans doublons).
    fn is_valid(&self) -> bool {
        for i in 0..9 {
            let mut row: Vec<u8> = Vec::new();
            for c in 0..9 {
                if self.cells[i][c] != 0 {
                    row.push(self.cells[i][c]);
                }
            }
            let mut vus_ligne = row.clone();
            vus_ligne.sort_unstable();
            vus_ligne.dedup();
            if vus_ligne.len() != row.len() {
                return false;
            }
            let mut col: Vec<u8> = Vec::new();
            for r in 0..9 {
                if self.cells[r][i] != 0 {
                    col.push(self.cells[r][i]);
                }
            }
            let mut vus_col = col.clone();
            vus_col.sort_unstable();
            vus_col.dedup();
            if vus_col.len() != col.len() {
                return false;
            }
        }
        for br in 0..3 {
            for bc in 0..3 {
                let mut block: Vec<u8> = Vec::new();
                for r in (br * 3)..(br * 3 + 3) {
                    for c in (bc * 3)..(bc * 3 + 3) {
                        if self.cells[r][c] != 0 {
                            block.push(self.cells[r][c]);
                        }
                    }
                }
                let mut vus_bloc = block.clone();
                vus_bloc.sort_unstable();
                vus_bloc.dedup();
                if vus_bloc.len() != block.len() {
                    return false;
                }
            }
        }
        true
    }

    /// Verifie si la grille est complete (pas de 0).
    fn is_complete(&self) -> bool {
        (0..9).all(|r| (0..9).all(|c| self.cells[r][c] != 0))
    }
}

impl fmt::Display for SudokuGrid {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        let mut lignes: Vec<String> = Vec::new();
        for r in 0..9 {
            if r > 0 && r % 3 == 0 {
                lignes.push("-".repeat(21));
            }
            let mut row_str = String::new();
            for c in 0..9 {
                if c > 0 && c % 3 == 0 {
                    row_str.push_str("| ");
                }
                let val = self.cells[r][c];
                if val != 0 {
                    row_str.push_str(&val.to_string());
                } else {
                    row_str.push('.');
                }
                // Comme `__str__` de la source, chaque case est suivie d'une
                // espace, y compris la derniere case de chaque ligne.
                row_str.push(' ');
            }
            lignes.push(row_str);
        }
        write!(f, "{}", lignes.join("\n"))
    }
}

fn load_puzzles(filepath: &Path, max_puzzles: Option<usize>) -> Result<Vec<String>, String> {
    let contenu = std::fs::read_to_string(filepath)
        .map_err(|e| format!("Impossible de lire {}: {}", filepath.display(), e))?;
    let mut puzzles: Vec<String> = Vec::new();
    for ligne in contenu.lines() {
        let l = ligne.trim();
        if l.chars().count() >= 81 {
            puzzles.push(l.chars().take(81).collect());
            if let Some(m) = max_puzzles {
                if puzzles.len() >= m {
                    break;
                }
            }
        }
    }
    if puzzles.is_empty() {
        return Err(format!("Aucun puzzle valide dans {}", filepath.display()));
    }
    Ok(puzzles)
}

// Charger les puzzles. Toute erreur est explicitement propagee : aucune grille
// vide ni grille de substitution ne peut etre presentee comme le corpus.
let notebook_dir = std::env::current_dir().unwrap_or_else(|_| std::path::PathBuf::from("."));
let puzzles_dir = notebook_dir.join("Puzzles");
let resultat: Result<(Vec<String>, SudokuGrid), String> = (|| {
    let easy_puzzles = load_puzzles(&puzzles_dir.join("Sudoku_Easy51.txt"), Some(5))?;
    let premiere = easy_puzzles
        .first()
        .ok_or_else(|| "Le corpus ne contient aucun puzzle".to_string())?;
    let test_grid = SudokuGrid::from_string(premiere)?;
    Ok((easy_puzzles, test_grid))
})();

match resultat {
    Ok((easy_puzzles, test_grid)) => {
        println!("Puzzles charges: {}", easy_puzzles.len());
        println!();
        println!("Grille de test:");
        println!("{}", test_grid);
    }
    Err(erreur) => {
        eprintln!("Erreur de chargement du corpus: {}", erreur);
    }
};

Puzzles charges: 5


Grille de test:


9 . 2 | . . 5 | 4 . 3 


1 . . | . 6 3 | . 2 5 


5 . 8 | 4 . 7 | . 6 . 


---------------------


... (3 outputs left out)

---------------------


2 4 . | 5 3 . | 6 . . 


7 . 5 | 2 . . | 3 . 4 


. 8 . | . 4 1 | 9 5 . 


### Lecture du corpus et structure de données

La sortie confirme **5 puzzles chargés**, et la grille de test affichée est la **grille témoin de la série** — comptez les indices : 9,2,5,4,3 en ligne 1, puis 1,6,3,2,5, 5,8,4,7,6... soit **45 indices** au total, exactement le même puzzle que Sudoku-04 (recuit), Sudoku-06-C# (MAC) et Sudoku-11 (Choco). Cette uniformité est la colonne vertébrale des comparaisons de la série : les mesures des sections suivantes porteront sur des entrées identiques. Les 51 puzzles faciles du dossier commun nourriront le benchmark de la section suivante.

La structure `SudokuGrid` qui produit cette sortie fournit la représentation des grilles :

| Aspect | Valeur | Signification |
|--------|--------|-------------|
| **Représentation** | Tableau `[[u8; 9]; 9]` | Structure naturelle pour accéder aux cases |
| **Cellules vides** | Valeur 0 | Convention standard pour les cases non remplies |
| **Validation** | 3 niveaux (ligne, colonne, bloc) | Vérification complète de la consistance |
| **Puzzles chargés** | 5 | Echantillon pour les tests |

**Points clés** :
1. La méthode `from_string()` permet de convertir une chaîne de 81 caractères en grille
2. La méthode `is_valid()` vérifie les contraintes Sudoku sans doublons
3. La méthode `clone()` crée une copie indépendante pour le backtracking
4. L'affichage formate avec `Display` separe visuellement les blocs 3x3

> **Note technique** : le tableau `[[u8; 9]; 9]` porte l'invariant 9×9 dans le type lui-même — il est impossible d'y loger une grille mal dimensionnée — et l'accès à n'importe quelle case reste en O(1), ce qui facilite le parcours des lignes, colonnes et blocs.

## Exercice : Calcul du domaine initial d'une cellule

### Objectif

Implémentez la fonction `get_cell_domain` qui calcule l'ensemble des valeurs possibles pour une cellule donnée d'une grille de Sudoku, en appliquant les contraintes de ligne, colonne et bloc.

Cette fonction est fondamentale : elle réalise manuellement ce que le CSP builder fera automatiquement plus tard. Comprendre cette étape est essentiel pour appréhender la reduction de domaine au cœur de la résolution par contraintes.

> **Étapes :**

1. **Collecter** les valeurs déjà presentes dans la même ligne que la cellule `(row, col)`
2. **Collecter** les valeurs de la même colonne
3. **Collecter** les valeurs du même bloc 3x3
4. **Retourner** l'ensemble `{1, ..., 9}` prive de toutes les valeurs collectées

> **Indices :**

- Si la cellule contient déjà une valeur non-nulle, son domaine est `{valeur}`
- Utilisez un `BTreeSet<u8>` pour eliminer les doublons entre ligne, colonne et bloc
- Testez sur la grille de référence : la cellule `(0, 1)` (première ligne, deuxième colonne) ne peut pas contenir 9 (déjà en ligne), ni 1, 5 (déjà en colonne)

In [5]:
// Stub C.1 : fn get_cell_domain(...)
use std::collections::BTreeSet;
use std::path::Path;

/// Representation d'une grille de Sudoku 9x9.
#[derive(Clone)]
struct SudokuGrid {
    cells: [[u8; 9]; 9],
}

impl SudokuGrid {
    fn from_string(s: &str) -> Result<Self, String> {
        let nettoye: String = s
            .chars()
            .map(|c| if c == '.' { '0' } else { c })
            .filter(|c| *c != ' ' && *c != '\n' && *c != '\r')
            .collect();
        if nettoye.chars().count() != 81 {
            return Err("La chaine doit avoir 81 caracteres".to_string());
        }
        let mut grille = SudokuGrid { cells: [[0u8; 9]; 9] };
        for (i, c) in nettoye.chars().enumerate() {
            let v: u8 = c
                .to_digit(10)
                .ok_or_else(|| "La chaine doit contenir uniquement des chiffres".to_string())? as u8;
            grille.cells[i / 9][i % 9] = v;
        }
        Ok(grille)
    }
}

fn load_puzzles(filepath: &Path, max_puzzles: Option<usize>) -> Result<Vec<String>, String> {
    let contenu = std::fs::read_to_string(filepath)
        .map_err(|e| format!("Impossible de lire {}: {}", filepath.display(), e))?;
    let mut puzzles: Vec<String> = Vec::new();
    for ligne in contenu.lines() {
        let l = ligne.trim();
        if l.chars().count() >= 81 {
            puzzles.push(l.chars().take(81).collect());
            if let Some(m) = max_puzzles {
                if puzzles.len() >= m {
                    break;
                }
            }
        }
    }
    if puzzles.is_empty() {
        return Err(format!("Aucun puzzle valide dans {}", filepath.display()));
    }
    Ok(puzzles)
}

/// Calcule le domaine (valeurs possibles) d'une cellule.
///
/// grid : grille de Sudoku
/// row : indice de ligne (0-8)
/// col : indice de colonne (0-8)
///
/// Rend l'ensemble des valeurs possibles pour la cellule.
fn get_cell_domain(_grid: &SudokuGrid, _row: usize, _col: usize) -> BTreeSet<u8> {
    // TODO etudiant : implémenter le calcul du domaine
    // Étape 1 : si la cellule est déjà remplie, retourner {valeur}
    // Étape 2 : collecter les valeurs interdites (ligne + colonne + bloc)
    // Étape 3 : retourner {1..9} - interdites
    (1u8..=9).collect() // TODO etudiant : remplacer par le calcul reel
}

/// Formate un ensemble de valeurs comme le fait le `repr` d'un `set` Python :
/// "{1, 2, 3}". Le formatage est explicite, jamais le `Debug` du conteneur.
fn formater_set(domain: &BTreeSet<u8>) -> String {
    let elements: Vec<String> = domain.iter().map(|v| v.to_string()).collect();
    format!("{{{}}}", elements.join(", "))
}

// Test sur la grille de reference
let notebook_dir = std::env::current_dir().unwrap_or_else(|_| std::path::PathBuf::from("."));
let puzzles_dir = notebook_dir.join("Puzzles");
let resultat: Result<BTreeSet<u8>, String> = (|| {
    let easy_puzzles = load_puzzles(&puzzles_dir.join("Sudoku_Easy51.txt"), Some(5))?;
    let premiere = easy_puzzles
        .first()
        .ok_or_else(|| "Le corpus ne contient aucun puzzle".to_string())?;
    let test_grid = SudokuGrid::from_string(premiere)?;
    Ok(get_cell_domain(&test_grid, 0, 1))
})();

match resultat {
    Ok(domain) => {
        println!("Domaine de la cellule (0,1) : {}", formater_set(&domain));
        println!("Taille du domaine : {} valeurs possibles", domain.len());
        println!("Exercice a completer");
    }
    Err(erreur) => {
        eprintln!("Erreur de chargement du corpus: {}", erreur);
    }
};

Domaine de la cellule (0,1) : {1, 2, 3, 4, 5, 6, 7, 8, 9}


Taille du domaine : 9 valeurs possibles


Exercice a completer


### Lecture honnête du domaine affiché : le placeholder de l'exercice

Lisez la sortie avec attention : le domaine affiché `{1, 2, ..., 9}` pour la cellule (0,1) est le **placeholder du stub** — la fonction rend `(1..=9).collect()` tant que l'exercice n'est pas complété, et la ligne « Exercice a completer » le signale (citation exacte de la sortie). Le **vrai** calcul exclurait les valeurs déjà posées dans le voisinage de (0,1) : la ligne 0 contient déjà 9, 2, 5, 4, 3, la colonne 1 contient 1, 5, 2, 9, 4, 8, et le bloc supérieur gauche d'autres valeurs encore — le domaine réel est donc nettement plus petit que 9 valeurs. C'est précisément l'objet de travail du CSP : le **domaine** d'une variable (ici, l'ensemble des valeurs légales d'une cellule) est la donnée que chaque heuristique de ce notebook interrogera — MRV trie par taille de domaine, le forward checking le rogne, AC-3 le vide jusqu'au point fixe.

## 2. Classe CSP Générique

Nous définissons une classe CSP générique inspiree du livre AIMA. Cette classe represente un **CSP binaire** (contraintes entre paires de variables).

Le qualificatif **binaire** est structurel : chaque contrainte relie exactement deux variables. La contrainte « AllDifferent » d'une ligne de Sudoku — les 9 cases toutes différentes — ne s'exprime pas directement dans ce cadre : elle se **décompose en paires**, une contrainte `val1 != val2` par paire de voisines. C'est ce que fera la section 3 : chaque case a 20 voisines (8 dans sa ligne, 8 dans sa colonne, 4 dans son bloc), soit 81 × 20 / 2 = **810 contraintes binaires** pour toute la grille. Toute la machinerie qui suit (AC-3, forward checking, MAC) raisonne sur ces paires, jamais sur la ligne entière.

L'interface tient en quatre données : `variables` (une par case de la grille 9 × 9), `domains` (les valeurs encore légales de chaque variable), `neighbors` (la liste d'adjacence des paires contraintes) et la contrainte binaire (le prédicat `val1 != val2`, ici la simple différence — en Rust elle est appliquée directement dans le code plutôt que stockée comme un champ `constraint_func`). Deux compteurs publics, `num_assignments` et `num_backtracks`, sont incrémentés par les solveurs : ce sont eux qui alimentent le benchmark de la section 10. Ce sont des compteurs d'**opérations algorithmiques**, indépendants de la vitesse de la machine — donc comparables entre le Python d'ici et le jumeau C# de la série, quand les temps d'exécution ne se comparent qu'en ordre de grandeur.

In [6]:
// struct Csp
use std::collections::BTreeMap;

/// Probleme de Satisfaction de Contraintes (CSP) binaire.
///
/// Inspire de AIMA - Russell & Norvig, Chapitre 6.
///
/// Les variables sont identifiees par un indice row-major unique
/// `v = r * 9 + c` (la source Python indexe par le couple (row, col) :
/// le port conserve le couple dans `variables` et utilise l'indice
/// row-major comme cle des assignations et des domaines).
struct Csp {
    variables: Vec<(usize, usize)>,
    domains: Vec<Vec<u8>>,
    neighbors: Vec<Vec<usize>>,
    // Compteurs pour l'analyse
    num_assignments: u64,
    num_backtracks: u64,
}

/// Indice row-major d'une case (r, c).
fn v_index(r: usize, c: usize) -> usize {
    r * 9 + c
}

impl Csp {
    /// Verifie si (var, val) est consistant avec l'assignation partielle.
    /// Contrainte binaire : deux voisins doivent porter des valeurs differentes.
    fn is_consistent(&self, var: usize, val: u8, assignment: &BTreeMap<usize, u8>) -> bool {
        for &voisin in &self.neighbors[var] {
            if let Some(&valeur_voisin) = assignment.get(&voisin) {
                if val == valeur_voisin {
                    return false;
                }
            }
        }
        true
    }

    /// Verifie si l'assignation est complete.
    fn is_complete(&self, assignment: &BTreeMap<usize, u8>) -> bool {
        assignment.len() == self.variables.len()
    }

    /// Retourne une copie profonde des domaines.
    fn copy_domains(&self) -> Vec<Vec<u8>> {
        self.domains.clone()
    }

    /// Retourne tous les arcs (Xi, Xj) du CSP.
    fn get_arcs(&self) -> Vec<(usize, usize)> {
        let mut arcs: Vec<(usize, usize)> = Vec::new();
        for var in 0..self.variables.len() {
            for &voisin in &self.neighbors[var] {
                arcs.push((var, voisin));
            }
        }
        arcs
    }
}

println!("Classe CSP definie.");

Classe CSP definie.


## 3. Construction du CSP Sudoku

Nous transformons une grille Sudoku en instance CSP avec :
- **81 variables** : une par cellule (0,0) à (8,8)
- **Domaines** : {1..9} pour les cellules vides, {v} pour les cellules fixées
- **Contraintes** : AllDifferent representee comme paires binaires !=

Deux choix de construction méritent d'être lus dans le code. **Les cellules fixées reçoivent un domaine singleton {v}** : l'énoncé entre dans le modèle comme un domaine déjà réduit, pas comme une contrainte séparée. Les solveurs n'ont ainsi besoin d'aucun cas particulier pour les indices posés — MRV les sélectionnera naturellement en premier (domaine de taille 1 = plus petit domaine restant), et les poser ne coûte qu'un essai.

**Le calcul des voisins** parcourt ligne, colonne puis bloc 3×3 et réunit les trois ensembles dans un `BTreeSet` : les recouvrements (un voisin de ligne peut aussi être voisin de bloc) sont dédupliqués, d'où exactement 20 voisins par cellule et non 8 + 8 + 8 = 24. Le `BTreeSet` étant trié, l'ordre de voisinage retenu est l'ordre croissant row-major — un ordre déterministe, là où l'ordre d'itération d'un `set` CPython n'est pas spécifié. La contrainte appliquée est la même pour tout le graphe — deux voisins doivent porter des valeurs différentes. C'est la seule contrainte du problème : toute la difficulté du Sudoku est dans la **topologie** (qui contraint qui), pas dans la nature des contraintes.

In [7]:
// fn build_csp / fn apply_solution
use std::collections::{BTreeMap, BTreeSet};

/// Representation d'une grille de Sudoku 9x9.
#[derive(Clone)]
struct SudokuGrid {
    cells: [[u8; 9]; 9],
}

/// Indice row-major d'une case (r, c).
fn v_index(r: usize, c: usize) -> usize {
    r * 9 + c
}

/// Probleme de Satisfaction de Contraintes (CSP) binaire.
struct Csp {
    variables: Vec<(usize, usize)>,
    domains: Vec<Vec<u8>>,
    neighbors: Vec<Vec<usize>>,
    num_assignments: u64,
    num_backtracks: u64,
}

/// Constructeur de CSP a partir d'une grille Sudoku.
fn build_csp(grid: &SudokuGrid) -> Csp {
    // Variables : (row, col) pour chaque cellule
    let mut variables: Vec<(usize, usize)> = Vec::new();
    for i in 0..9 {
        for j in 0..9 {
            variables.push((i, j));
        }
    }

    // Domaines : 1-9 pour les vides, valeur unique pour les fixées
    let mut domains: Vec<Vec<u8>> = vec![Vec::new(); 81];
    for i in 0..9 {
        for j in 0..9 {
            let value = grid.cells[i][j];
            domains[v_index(i, j)] = if value == 0 {
                (1u8..=9).collect()
            } else {
                vec![value]
            };
        }
    }

    // Voisins : même ligne, même colonne, même bloc
    let mut neighbors: Vec<Vec<usize>> = vec![Vec::new(); 81];
    for i in 0..9 {
        for j in 0..9 {
            let mut neighbor_set: BTreeSet<usize> = BTreeSet::new();

            // Même ligne
            for k in 0..9 {
                if k != j {
                    neighbor_set.insert(v_index(i, k));
                }
            }

            // Même colonne
            for k in 0..9 {
                if k != i {
                    neighbor_set.insert(v_index(k, j));
                }
            }

            // Même bloc 3x3
            let block_row = (i / 3) * 3;
            let block_col = (j / 3) * 3;
            for r in block_row..(block_row + 3) {
                for c in block_col..(block_col + 3) {
                    if r != i || c != j {
                        neighbor_set.insert(v_index(r, c));
                    }
                }
            }

            neighbors[v_index(i, j)] = neighbor_set.into_iter().collect();
        }
    }

    // Fonction de contrainte : valeurs différentes
    // (en Rust, la contrainte binaire `val1 != val2` est appliquee dans le code)

    Csp {
        variables,
        domains,
        neighbors,
        num_assignments: 0,
        num_backtracks: 0,
    }
}

/// Applique une solution CSP a une grille Sudoku.
fn apply_solution(grid: &mut SudokuGrid, assignment: &BTreeMap<usize, u8>) {
    for (&v, &value) in assignment.iter() {
        grid.cells[v / 9][v % 9] = value;
    }
}

println!("SudokuCSPBuilder defini.");

SudokuCSPBuilder defini.


## 4. Backtracking Simple

La première stratégie est la recherche chronologique dans toute sa nudité : prendre la **première variable non assignée** dans l'ordre de la liste (balayage ligne par ligne, de (0,0) vers (8,8)), essayer ses valeurs dans l'ordre 1, 2, 3…, et quand une valeur passe le test de consistance (`is_consistent` interroge les voisins déjà assignés), descendre d'un cran ; quand toutes les valeurs échouent, remonter d'un cran et essayer la valeur suivante de la variable précédente. Aucune mémoire des conflits, aucune réduction de domaine : l'algorithme ne sait pas *pourquoi* une branche est morte, seulement *qu'elle* l'est.

Le coût de cette indifférence se chiffrera dans le benchmark de la section 10 : sur la grille témoin à 45 indices, ce solveur dépassera **2,8 millions d'assignations** là où les versions outillées en feront 255 ou 81. La seule intelligence ici est le test de consistance — sans lui, l'énumération exhaustive des 9^36 remplissages possibles des 36 cases vides ne terminerait pas.

In [8]:
// fn solve (backtracking simple)
use std::collections::BTreeMap;

/// Probleme de Satisfaction de Contraintes (CSP) binaire.
struct Csp {
    variables: Vec<(usize, usize)>,
    domains: Vec<Vec<u8>>,
    neighbors: Vec<Vec<usize>>,
    num_assignments: u64,
    num_backtracks: u64,
}

/// Indice row-major d'une case (r, c).
fn v_index(r: usize, c: usize) -> usize {
    r * 9 + c
}

impl Csp {
    /// Verifie si (var, val) est consistant avec l'assignation partielle.
    fn is_consistent(&self, var: usize, val: u8, assignment: &BTreeMap<usize, u8>) -> bool {
        for &voisin in &self.neighbors[var] {
            if let Some(&valeur_voisin) = assignment.get(&voisin) {
                if val == valeur_voisin {
                    return false;
                }
            }
        }
        true
    }

    /// Verifie si l'assignation est complete.
    fn is_complete(&self, assignment: &BTreeMap<usize, u8>) -> bool {
        assignment.len() == self.variables.len()
    }
}

/// Backtracking simple pour CSP.
/// Rend true si une assignation complete a ete trouvee, false sinon.
fn backtracking_simple_solve(csp: &mut Csp, assignment: &mut BTreeMap<usize, u8>) -> bool {
    if csp.is_complete(assignment) {
        return true;
    }

    // Choisir la première variable non assignee (ordre naif)
    let var = match csp
        .variables
        .iter()
        .map(|&(r, c)| v_index(r, c))
        .find(|v| !assignment.contains_key(v))
    {
        Some(v) => v,
        None => return true,
    };

    let valeurs = csp.domains[var].clone();
    for val in valeurs {
        csp.num_assignments += 1;

        if csp.is_consistent(var, val, assignment) {
            assignment.insert(var, val);
            if backtracking_simple_solve(csp, assignment) {
                return true;
            }
            assignment.remove(&var);
            csp.num_backtracks += 1;
        }
    }

    false
}

println!("BacktrackingSimple defini.");

BacktrackingSimple defini.


## 5. Heuristiques : MRV et LCV

### MRV (Minimum Remaining Values)
Heuristique de **sélection de variable** : choisir la variable avec le plus petit domaine restant.

### LCV (Least Constraining Value)
Heuristique d'**ordonnancement des valeurs** : essayer d'abord la valeur qui élimine le moins de possibilites chez les voisins.

Dans cette implémentation, les deux heuristiques vivent dans les fonctions `select_mrv` et `order_lcv` et sont **orthogonales par construction** : MRV choisit *la prochaine variable à assigner* (celle au domaine le plus petit — révéler tôt les conflits, quand annuler une branche coûte peu), LCV ordonne *les valeurs à essayer* pour la variable choisie (commencer par celle qui contraint le moins les voisines — garder des options ouvertes). Aucune des deux ne réduit l'arbre en soi : elles réordonnent l'exploration. Le benchmark de la section 10 chiffre la hiérarchie complète : le backtracking seul explose à 2 889 322 assignations, MRV+LCV le ramène à 255, et dès lors que la propagation (FC, MAC) entre en jeu, les 81 assignations exactes suffisent.

In [9]:
// fn select_mrv / fn order_lcv
mod csp_heuristics_17 {
    use std::cmp::Reverse;
    use std::collections::BTreeMap;

    /// Probleme de Satisfaction de Contraintes (CSP) binaire.
    struct Csp {
        variables: Vec<(usize, usize)>,
        domains: Vec<Vec<u8>>,
        neighbors: Vec<Vec<usize>>,
        num_assignments: u64,
        num_backtracks: u64,
    }

    /// Indice row-major d'une case (r, c).
    fn v_index(r: usize, c: usize) -> usize {
        r * 9 + c
    }

    impl Csp {
        /// Verifie si (var, val) est consistant avec l'assignation partielle.
        fn is_consistent(&self, var: usize, val: u8, assignment: &BTreeMap<usize, u8>) -> bool {
            for &voisin in &self.neighbors[var] {
                if let Some(&valeur_voisin) = assignment.get(&voisin) {
                    if val == valeur_voisin {
                        return false;
                    }
                }
            }
            true
        }
    }

    /// MRV : Selectionne la variable avec le moins de valeurs viables.
    ///
    /// En cas d'egalite, utilise le degre (nombre de voisins non assignes).
    /// Les egalites sont tranchees par la PREMIERE variable dans l'ordre
    /// row-major des variables (parcours lineaire, comparaison stricte).
    fn select_mrv(
        csp: &Csp,
        assignment: &BTreeMap<usize, u8>,
        current_domains: &[Vec<u8>],
    ) -> usize {
        let mut meilleure: Option<(usize, Reverse<usize>, usize)> = None;
        for &(r, c) in &csp.variables {
            let v = v_index(r, c);
            if assignment.contains_key(&v) {
                continue;
            }
            let remaining = current_domains[v]
                .iter()
                .filter(|&&val| csp.is_consistent(v, val, assignment))
                .count();
            let degree = csp.neighbors[v]
                .iter()
                .filter(|n| !assignment.contains_key(n))
                .count();
            // MRV croissant, puis degre decroissant
            let candidat = (remaining, Reverse(degree), v);
            match meilleure {
                None => meilleure = Some(candidat),
                Some(ref m) if candidat < *m => meilleure = Some(candidat),
                _ => {}
            }
        }
        meilleure.map(|(_, _, v)| v).unwrap_or(0)
    }

    /// LCV : Ordonne les valeurs par nombre de conflits croissant.
    /// Le tri est STABLE : a egalite de conflits, l'ordre du domaine est conserve.
    fn order_lcv(
        csp: &Csp,
        var: usize,
        assignment: &BTreeMap<usize, u8>,
        current_domains: &[Vec<u8>],
    ) -> Vec<u8> {
        let mut conflits: Vec<usize> = Vec::new();
        for &val in &current_domains[var] {
            let mut count = 0usize;
            for &voisin in &csp.neighbors[var] {
                if !assignment.contains_key(&voisin) {
                    for &nval in &current_domains[voisin] {
                        if val == nval {
                            count += 1;
                        }
                    }
                }
            }
            conflits.push(count);
        }
        let mut paires: Vec<(u8, usize)> = current_domains[var]
            .iter()
            .copied()
            .zip(conflits.iter().copied())
            .collect();
        paires.sort_by_key(|&(_, c)| c);
        paires.into_iter().map(|(v, _)| v).collect()
    }

    pub fn run() {
        println!("CSPHeuristics defini.");
    }
}

csp_heuristics_17::run();

CSPHeuristics defini.


## 6. Backtracking Ameliore (MRV + LCV)

Le squelette récursif est identique au backtracking simple — même descente, même remontée — mais les deux décisions aveugles deviennent des heuristiques : *quelle* variable traiter (MRV plutôt que « la première de la liste ») et *dans quel ordre* essayer ses valeurs (LCV plutôt que l'ordre 1, 2, 3…). Un détail d'implémentation compte : les heuristiques lisent `current_domains`, une **copie** des domaines (via `copy_domains()`) et non l'état partagé du CSP — la remontée doit retrouver intacts les domaines que la descente avait laissés.

Ce solveur ne modifie donc rien : il ne fait que **réordonner**. Toute la propagation reste à charge des sections suivantes (Forward Checking, MAC). D'où le verdict chiffré du benchmark : 255 assignations et zéro backtrack sur la grille témoin, contre plus de 2,8 millions d'assignations pour la version nue — un facteur quatre chiffres obtenu sans une seule élimination de valeur, rien qu'en choisissant mieux l'ordre d'attaque.

In [10]:
// fn solve (MRV + LCV)
use std::cmp::Reverse;
use std::collections::BTreeMap;

/// Probleme de Satisfaction de Contraintes (CSP) binaire.
struct Csp {
    variables: Vec<(usize, usize)>,
    domains: Vec<Vec<u8>>,
    neighbors: Vec<Vec<usize>>,
    num_assignments: u64,
    num_backtracks: u64,
}

/// Indice row-major d'une case (r, c).
fn v_index(r: usize, c: usize) -> usize {
    r * 9 + c
}

impl Csp {
    fn is_consistent(&self, var: usize, val: u8, assignment: &BTreeMap<usize, u8>) -> bool {
        for &voisin in &self.neighbors[var] {
            if let Some(&valeur_voisin) = assignment.get(&voisin) {
                if val == valeur_voisin {
                    return false;
                }
            }
        }
        true
    }

    fn is_complete(&self, assignment: &BTreeMap<usize, u8>) -> bool {
        assignment.len() == self.variables.len()
    }

    /// Retourne une copie profonde des domaines.
    fn copy_domains(&self) -> Vec<Vec<u8>> {
        self.domains.clone()
    }
}

/// MRV : Selectionne la variable avec le moins de valeurs viables.
fn select_mrv(csp: &Csp, assignment: &BTreeMap<usize, u8>, current_domains: &[Vec<u8>]) -> usize {
    let mut meilleure: Option<(usize, Reverse<usize>, usize)> = None;
    for &(r, c) in &csp.variables {
        let v = v_index(r, c);
        if assignment.contains_key(&v) {
            continue;
        }
        let remaining = current_domains[v]
            .iter()
            .filter(|&&val| csp.is_consistent(v, val, assignment))
            .count();
        let degree = csp.neighbors[v]
            .iter()
            .filter(|n| !assignment.contains_key(n))
            .count();
        let candidat = (remaining, Reverse(degree), v);
        match meilleure {
            None => meilleure = Some(candidat),
            Some(ref m) if candidat < *m => meilleure = Some(candidat),
            _ => {}
        }
    }
    meilleure.map(|(_, _, v)| v).unwrap_or(0)
}

/// LCV : Ordonne les valeurs par nombre de conflits croissant (tri stable).
fn order_lcv(csp: &Csp, var: usize, assignment: &BTreeMap<usize, u8>, current_domains: &[Vec<u8>]) -> Vec<u8> {
    let mut conflits: Vec<usize> = Vec::new();
    for &val in &current_domains[var] {
        let mut count = 0usize;
        for &voisin in &csp.neighbors[var] {
            if !assignment.contains_key(&voisin) {
                for &nval in &current_domains[voisin] {
                    if val == nval {
                        count += 1;
                    }
                }
            }
        }
        conflits.push(count);
    }
    let mut paires: Vec<(u8, usize)> = current_domains[var]
        .iter()
        .copied()
        .zip(conflits.iter().copied())
        .collect();
    paires.sort_by_key(|&(_, c)| c);
    paires.into_iter().map(|(v, _)| v).collect()
}

/// Backtracking avec heuristiques MRV et LCV.
/// Rend true si une assignation complete a ete trouvee, false sinon.
fn backtracking_improved_solve(
    csp: &mut Csp,
    assignment: &mut BTreeMap<usize, u8>,
    current_domains: &mut Vec<Vec<u8>>,
    use_mrv: bool,
    use_lcv: bool,
) -> bool {
    if csp.is_complete(assignment) {
        return true;
    }

    // Sélection de variable
    let var = if use_mrv {
        select_mrv(csp, assignment, current_domains)
    } else {
        match csp
            .variables
            .iter()
            .map(|&(r, c)| v_index(r, c))
            .find(|v| !assignment.contains_key(v))
        {
            Some(v) => v,
            None => return true,
        }
    };

    // Ordonnancement des valeurs
    let valeurs = if use_lcv {
        order_lcv(csp, var, assignment, current_domains)
    } else {
        current_domains[var].clone()
    };

    for val in valeurs {
        csp.num_assignments += 1;

        if csp.is_consistent(var, val, assignment) {
            assignment.insert(var, val);
            if backtracking_improved_solve(csp, assignment, current_domains, use_mrv, use_lcv) {
                return true;
            }
            assignment.remove(&var);
            csp.num_backtracks += 1;
        }
    }

    false
}

println!("BacktrackingImproved defini.");

BacktrackingImproved defini.


## 7. Forward Checking

Le **Forward Checking** propage l'assignation d'une variable vers ses voisins immediats, reduisant leurs domaines et détectant les échecs plus tot.

La différence avec le backtracking amélioré est un changement de **moment**. Celui-ci vérifiait la consistance au moment d'assigner (« cette valeur contredit-elle un voisin déjà posé ? ») ; le forward checking regarde **en avant** : dès qu'une variable est posée, il retire cette valeur des domaines de tous ses voisins non assignés. La trace de ces retraits est conservée (`removals`, la liste des couples (voisin, valeur retirée)) pour pouvoir les **restaurer** à la remontée — le mécanisme de « trail » classique des solveurs à retour arrière.

Le gain décisif est le test `current_domains[voisin].is_empty()` : si la propagation vide le domaine d'un voisin, l'échec est déclaré **immédiatement**, sans attendre que la recherche atteigne cette variable — le cul-de-sac est détecté à distance, plusieurs niveaux avant qu'il ne bloque la descente. C'est ce décalage qui explique les 81 assignations sans aucun backtrack mesurées sur la grille témoin : chaque pose est pré-validée pour tout le voisinage d'un coup, et la recherche ne s'engage jamais dans une branche qu'une propagation à distance aurait pu condamner.

In [11]:
// propagate / restore / solve
use std::cmp::Reverse;
use std::collections::BTreeMap;

/// Probleme de Satisfaction de Contraintes (CSP) binaire.
struct Csp {
    variables: Vec<(usize, usize)>,
    domains: Vec<Vec<u8>>,
    neighbors: Vec<Vec<usize>>,
    num_assignments: u64,
    num_backtracks: u64,
}

/// Indice row-major d'une case (r, c).
fn v_index(r: usize, c: usize) -> usize {
    r * 9 + c
}

impl Csp {
    fn is_consistent(&self, var: usize, val: u8, assignment: &BTreeMap<usize, u8>) -> bool {
        for &voisin in &self.neighbors[var] {
            if let Some(&valeur_voisin) = assignment.get(&voisin) {
                if val == valeur_voisin {
                    return false;
                }
            }
        }
        true
    }

    fn is_complete(&self, assignment: &BTreeMap<usize, u8>) -> bool {
        assignment.len() == self.variables.len()
    }

    /// Retourne une copie profonde des domaines.
    fn copy_domains(&self) -> Vec<Vec<u8>> {
        self.domains.clone()
    }
}

/// MRV : Selectionne la variable avec le moins de valeurs viables.
fn select_mrv(csp: &Csp, assignment: &BTreeMap<usize, u8>, current_domains: &[Vec<u8>]) -> usize {
    let mut meilleure: Option<(usize, Reverse<usize>, usize)> = None;
    for &(r, c) in &csp.variables {
        let v = v_index(r, c);
        if assignment.contains_key(&v) {
            continue;
        }
        let remaining = current_domains[v]
            .iter()
            .filter(|&&val| csp.is_consistent(v, val, assignment))
            .count();
        let degree = csp.neighbors[v]
            .iter()
            .filter(|n| !assignment.contains_key(n))
            .count();
        let candidat = (remaining, Reverse(degree), v);
        match meilleure {
            None => meilleure = Some(candidat),
            Some(ref m) if candidat < *m => meilleure = Some(candidat),
            _ => {}
        }
    }
    meilleure.map(|(_, _, v)| v).unwrap_or(0)
}

/// LCV : Ordonne les valeurs par nombre de conflits croissant (tri stable).
fn order_lcv(csp: &Csp, var: usize, assignment: &BTreeMap<usize, u8>, current_domains: &[Vec<u8>]) -> Vec<u8> {
    let mut conflits: Vec<usize> = Vec::new();
    for &val in &current_domains[var] {
        let mut count = 0usize;
        for &voisin in &csp.neighbors[var] {
            if !assignment.contains_key(&voisin) {
                for &nval in &current_domains[voisin] {
                    if val == nval {
                        count += 1;
                    }
                }
            }
        }
        conflits.push(count);
    }
    let mut paires: Vec<(u8, usize)> = current_domains[var]
        .iter()
        .copied()
        .zip(conflits.iter().copied())
        .collect();
    paires.sort_by_key(|&(_, c)| c);
    paires.into_iter().map(|(v, _)| v).collect()
}

/// Forward Checking : propage l'assignation vers les voisins.
///
/// Propage l'assignation var=val vers les voisins non assignes.
/// Rend (removals, success) : la liste des valeurs retirees et le succes.
fn fc_propagate(
    csp: &Csp,
    var: usize,
    val: u8,
    assignment: &BTreeMap<usize, u8>,
    current_domains: &mut Vec<Vec<u8>>,
) -> (Vec<(usize, u8)>, bool) {
    let mut removals: Vec<(usize, u8)> = Vec::new();

    for &voisin in &csp.neighbors[var] {
        if !assignment.contains_key(&voisin) {
            let mut to_remove: Vec<u8> = Vec::new();
            for &nval in &current_domains[voisin] {
                if val == nval {
                    to_remove.push(nval);
                    removals.push((voisin, nval));
                }
            }

            for r in to_remove {
                current_domains[voisin].retain(|&x| x != r);
            }

            if current_domains[voisin].is_empty() {
                return (removals, false); // Domaine vide = échec
            }
        }
    }

    (removals, true)
}

/// Restaure les valeurs retirees.
/// Comme `list.append` en Python, les valeurs reviennent EN FIN de domaine.
fn fc_restore(current_domains: &mut Vec<Vec<u8>>, removals: &[(usize, u8)]) {
    for &(var, val) in removals {
        current_domains[var].push(val);
    }
}

/// Backtracking avec Forward Checking.
fn fc_solve(
    csp: &mut Csp,
    assignment: &mut BTreeMap<usize, u8>,
    current_domains: &mut Vec<Vec<u8>>,
) -> bool {
    if csp.is_complete(assignment) {
        return true;
    }

    let var = select_mrv(csp, assignment, current_domains);
    let valeurs = order_lcv(csp, var, assignment, current_domains);

    for val in valeurs {
        csp.num_assignments += 1;

        if csp.is_consistent(var, val, assignment) {
            assignment.insert(var, val);

            let (removals, success) = fc_propagate(csp, var, val, assignment, current_domains);

            if success && fc_solve(csp, assignment, current_domains) {
                return true;
            }

            fc_restore(current_domains, &removals);
            assignment.remove(&var);
            csp.num_backtracks += 1;
        }
    }

    false
}

println!("ForwardChecking defini.");

ForwardChecking defini.


## Exercice : Analyse pas-à-pas de la propagation Forward Checking

### Objectif

Implémentez la fonction `analyze_fc_propagation` qui simule et affiche les étapes de réduction de domaine lorsqu'on assigné une valeur à une cellule donnée, en suivant le mécanisme du Forward Checking.

Cet exercice vous permettra de visualiser concrètement comment le Forward Checking elague l'espace de recherche : à chaque assignation, les domaines des voisins sont réduits, et les domaines vides signalent un échec précoce.

> **Étapes :**

1. **Construire** le CSP à partir de la grille et copier les domaines (`build_csp(&grid)` puis `csp.copy_domains()`)
2. **Choisir** une cellule et une valeur (par exemple `(0, 1)` avec la valeur `6`)
3. **Appliquer** `fc_propagate()` pour obtenir les valeurs éliminées
4. **Afficher** pour chaque voisin affecté : les valeurs retirées et la taille du domaine restant
5. **Detecter** si la propagation a provoqué un domaine vide (échec)

> **Indices :**

- `fc_propagate(csp, var, val, assignment, current_domains)` retourne `(removals, success)` — c'est l'équivalent Rust de `ForwardChecking.propagate` de la source
- `removals` est un `Vec<(usize, u8)>` : le couple (voisin, valeur retirée)
- Utilisez un `BTreeMap<usize, Vec<u8>>` pour regrouper les retraits par voisin
- Comparez la taille des domaines avant et après propagation pour mesurer l'impact

In [12]:
// Stub C.1 : fn analyze_fc_propagation(...)
mod exercise_23 {
    use std::collections::{BTreeMap, BTreeSet};
    use std::path::Path;

    /// Representation d'une grille de Sudoku 9x9.
    #[derive(Clone)]
    struct SudokuGrid {
        cells: [[u8; 9]; 9],
    }

    impl SudokuGrid {
        fn from_string(s: &str) -> Result<Self, String> {
            let nettoye: String = s
                .chars()
                .map(|c| if c == '.' { '0' } else { c })
                .filter(|c| *c != ' ' && *c != '\n' && *c != '\r')
                .collect();
            if nettoye.chars().count() != 81 {
                return Err("La chaine doit avoir 81 caracteres".to_string());
            }
            let mut grille = SudokuGrid { cells: [[0u8; 9]; 9] };
            for (i, c) in nettoye.chars().enumerate() {
                let v: u8 = c
                    .to_digit(10)
                    .ok_or_else(|| "La chaine doit contenir uniquement des chiffres".to_string())? as u8;
                grille.cells[i / 9][i % 9] = v;
            }
            Ok(grille)
        }
    }

    fn load_puzzles(filepath: &Path, max_puzzles: Option<usize>) -> Result<Vec<String>, String> {
        let contenu = std::fs::read_to_string(filepath)
            .map_err(|e| format!("Impossible de lire {}: {}", filepath.display(), e))?;
        let mut puzzles: Vec<String> = Vec::new();
        for ligne in contenu.lines() {
            let l = ligne.trim();
            if l.chars().count() >= 81 {
                puzzles.push(l.chars().take(81).collect());
                if let Some(m) = max_puzzles {
                    if puzzles.len() >= m {
                        break;
                    }
                }
            }
        }
        if puzzles.is_empty() {
            return Err(format!("Aucun puzzle valide dans {}", filepath.display()));
        }
        Ok(puzzles)
    }

    /// Probleme de Satisfaction de Contraintes (CSP) binaire.
    struct Csp {
        variables: Vec<(usize, usize)>,
        domains: Vec<Vec<u8>>,
        neighbors: Vec<Vec<usize>>,
        num_assignments: u64,
        num_backtracks: u64,
    }

    impl Csp {
        fn is_consistent(&self, var: usize, val: u8, assignment: &BTreeMap<usize, u8>) -> bool {
            self.neighbors[var].iter().all(|&voisin| {
                assignment.get(&voisin).map_or(true, |&valeur| valeur != val)
            })
        }

        fn copy_domains(&self) -> Vec<Vec<u8>> {
            self.domains.clone()
        }
    }

    /// Construit le CSP necessaire a l'exercice.
    fn build_csp(grid: &SudokuGrid) -> Csp {
        let mut variables = Vec::new();
        let mut domains = vec![Vec::new(); 81];
        let mut neighbors = vec![Vec::new(); 81];

        for r in 0..9 {
            for c in 0..9 {
                let var = r * 9 + c;
                variables.push((r, c));
                domains[var] = if grid.cells[r][c] == 0 {
                    (1u8..=9).collect()
                } else {
                    vec![grid.cells[r][c]]
                };

                let mut neighbor_set = BTreeSet::new();
                for k in 0..9 {
                    if k != c {
                        neighbor_set.insert(r * 9 + k);
                    }
                    if k != r {
                        neighbor_set.insert(k * 9 + c);
                    }
                }
                let block_row = (r / 3) * 3;
                let block_col = (c / 3) * 3;
                for br in block_row..(block_row + 3) {
                    for bc in block_col..(block_col + 3) {
                        if br != r || bc != c {
                            neighbor_set.insert(br * 9 + bc);
                        }
                    }
                }
                neighbors[var] = neighbor_set.into_iter().collect();
            }
        }

        Csp {
            variables,
            domains,
            neighbors,
            num_assignments: 0,
            num_backtracks: 0,
        }
    }

    /// Infrastructure de propagation FC fournie a l'etudiant.
    fn fc_propagate(
        csp: &Csp,
        var: usize,
        val: u8,
        assignment: &BTreeMap<usize, u8>,
        current_domains: &mut [Vec<u8>],
    ) -> (Vec<(usize, u8)>, bool) {
        let mut removals = Vec::new();
        for &voisin in &csp.neighbors[var] {
            if !assignment.contains_key(&voisin) {
                let mut a_retirer = Vec::new();
                for &nval in &current_domains[voisin] {
                    if nval == val {
                        a_retirer.push(nval);
                        removals.push((voisin, nval));
                    }
                }
                for valeur in a_retirer {
                    current_domains[voisin].retain(|&x| x != valeur);
                }
                if current_domains[voisin].is_empty() {
                    return (removals, false);
                }
            }
        }
        (removals, true)
    }

    /// Analyse et affiche la reduction de domaine par Forward Checking.
    ///
    /// grid : Grille de Sudoku
    /// var : Variable (cellule) a assigner, sous forme (row, col)
    /// val : Valeur a assigner
    fn analyze_fc_propagation(_grid: &SudokuGrid, _var: (usize, usize), _val: u8) {
        // TODO etudiant : implémenter l'analyse de propagation FC
        // Étape 1 : construire le CSP avec SudokuCSPBuilder.build_csp(grid)
        // Étape 2 : copier les domaines avec csp.copy_domains()
        // Étape 3 : compter les valeurs de domaine avant propagation
        // Étape 4 : appeler ForwardChecking.propagate(csp, var, val, {}, current_domains)
        // Étape 5 : regrouper les retraits par voisin et afficher le detail
        // Indice : defaultdict(list) pour grouper les (neighbor, val_retiree) par voisin
    }

    pub fn run() {
        // Test sur la grille de référence
        let notebook_dir = std::env::current_dir().unwrap_or_else(|_| std::path::PathBuf::from("."));
        let puzzles_dir = notebook_dir.join("Puzzles");
        let resultat: Result<SudokuGrid, String> = (|| {
            let easy_puzzles = load_puzzles(&puzzles_dir.join("Sudoku_Easy51.txt"), Some(5))?;
            let premiere = easy_puzzles
                .first()
                .ok_or_else(|| "Le corpus ne contient aucun puzzle".to_string())?;
            SudokuGrid::from_string(premiere)
        })();

        match resultat {
            Ok(test_grid_fc) => {
                analyze_fc_propagation(&test_grid_fc, (0, 1), 6);
                println!("Exercice a completer");
            }
            Err(erreur) => {
                eprintln!("Erreur de chargement du corpus: {}", erreur);
            }
        }
    }
}

exercise_23::run();


Exercice a completer


## 8. Arc Consistency (AC-3)

L'algorithme **AC-3** assure que pour chaque arc (Xi, Xj), toute valeur de Xi a un support dans Xj.

Le moteur d'AC-3 tient en trois pièces : une **file d'arcs** (initialisée avec tous les couples (Xi, Xj) voisins), l'opération **revise** (pour chaque valeur de Di, existerait-il une valeur support dans Dj ? sinon, retirer la valeur de Di), et la détection du **point fixe** (si Di a changé, remettre en file tous les arcs pointant vers Xi — l'élimination se propage). Le pré-traitement complet vide la file une fois pour toutes ; MAC, en section 9, repeuplera une mini-file à chaque assignation. C'est ce même mécanisme que le notebook Sudoku-14-BDD compile sous forme d'automate, et que la propagation de Norvig (Sudoku-07) réalise par paires de cellules.

Le coût de l'algorithme se lit dans sa boucle : chaque sortie de file déclenche un `revise`, qui pour chaque valeur de Di balaie Dj à la recherche d'un support — d'où la borne classique O(e·d³) pour e arcs et des domaines de taille d. Sur le Sudoku (810 arcs, domaines de taille au plus 9), le pré-traitement complet reste quasi instantané ; c'est sa répétition à chaque nœud de recherche qui motivera le compromis FC/MAC que le benchmark de la section 10 mesurera.

In [13]:
// revise / run
use std::collections::VecDeque;

/// Probleme de Satisfaction de Contraintes (CSP) binaire.
/// Les variables sont identifiees par un indice row-major v = r * 9 + c.
struct Csp {
    variables: Vec<(usize, usize)>,
    domains: Vec<Vec<u8>>,
    neighbors: Vec<Vec<usize>>,
    num_assignments: u64,
    num_backtracks: u64,
}

impl Csp {
    /// Retourne tous les arcs (Xi, Xj) du CSP.
    fn get_arcs(&self) -> Vec<(usize, usize)> {
        let mut arcs: Vec<(usize, usize)> = Vec::new();
        for var in 0..self.variables.len() {
            for &voisin in &self.neighbors[var] {
                arcs.push((var, voisin));
            }
        }
        arcs
    }
}

/// Rend l'arc (xi, xj) arc-consistent.
///
/// Retourne true si le domaine de xi a ete modifie.
fn ac3_revise(
    csp: &Csp,
    xi: usize,
    xj: usize,
    current_domains: &mut [Vec<u8>],
) -> bool {
    let mut revised = false;
    let mut to_remove: Vec<u8> = Vec::new();

    for &val_i in &current_domains[xi] {
        let has_support = current_domains[xj]
            .iter()
            .any(|&val_j| val_i != val_j);
        if !has_support {
            to_remove.push(val_i);
            revised = true;
        }
    }

    for val in to_remove {
        current_domains[xi].retain(|&x| x != val);
    }

    revised
}

/// Rend le CSP arc-consistent.
///
/// Retourne false si un domaine devient vide (echec).
fn ac3_run(
    csp: &Csp,
    current_domains: &mut [Vec<u8>],
    arcs: Option<Vec<(usize, usize)>>,
) -> bool {
    // La source utilise `deque(arcs if arcs else csp.get_arcs())` :
    // `None` et une liste vide entrainent tous les arcs du CSP.
    let arcs_initiaux = match arcs {
        Some(arcs) if !arcs.is_empty() => arcs,
        _ => csp.get_arcs(),
    };
    let mut queue: VecDeque<(usize, usize)> = arcs_initiaux.into_iter().collect();

    while let Some((xi, xj)) = queue.pop_front() {
        if ac3_revise(csp, xi, xj, current_domains) {
            if current_domains[xi].is_empty() {
                return false;
            }
            for &xk in &csp.neighbors[xi] {
                if xk != xj {
                    queue.push_back((xk, xi));
                }
            }
        }
    }

    true
}

println!("AC3 defini.");


AC3 defini.


## Exercice : Pre-processing par arc-consistance

### Objectif

Implémentez la fonction `preprocess_ac3` qui utilise AC-3 comme pre-traitement avant la résolution. L'objectif est de mesurer combien de valeurs de domaine sont éliminées par la seule arc-consistance, sans lancer la recherche.

Ce pre-traitement est une étape clef des solveurs CSP performants : en reduisant les domaines avant la recherche, on diminue drastiquement l'espace à explorer.

> **Étapes :**

1. **Construire** le CSP à partir d'une grille
2. **Copier** les domaines initiaux (compter le nombre total de valeurs)
3. **Appliquer** `ac3_run()` sur les domaines copies
4. **Compter** le nombre de valeurs éliminées et afficher les statistiques
5. **Résoudre** ensuite avec MAC et comparer les performances avec/sans pre-processing

> **Indices :**

- Le nombre total de valeurs initiales = `csp.domains.iter().map(|d| d.len()).sum::<usize>()`
- `ac3_run()` modifie `current_domains` en place et retourne un booléen (succès/échec)
- Sur une grille facile, le pre-processing AC-3 seul peut eliminer > 50% des valeurs de domaine

In [14]:
// Stub C.1 : fn preprocess_ac3(...)
mod exercise_27 {
    use std::collections::{BTreeSet, VecDeque};
    use std::path::Path;

    /// Representation d'une grille de Sudoku 9x9.
    #[derive(Clone)]
    struct SudokuGrid {
        cells: [[u8; 9]; 9],
    }

    impl SudokuGrid {
        fn from_string(s: &str) -> Result<Self, String> {
            let nettoye: String = s
                .chars()
                .map(|c| if c == '.' { '0' } else { c })
                .filter(|c| *c != ' ' && *c != '\n' && *c != '\r')
                .collect();
            if nettoye.chars().count() != 81 {
                return Err("La chaine doit avoir 81 caracteres".to_string());
            }
            let mut grille = SudokuGrid { cells: [[0u8; 9]; 9] };
            for (i, c) in nettoye.chars().enumerate() {
                let v: u8 = c
                    .to_digit(10)
                    .ok_or_else(|| "La chaine doit contenir uniquement des chiffres".to_string())? as u8;
                grille.cells[i / 9][i % 9] = v;
            }
            Ok(grille)
        }
    }

    fn load_puzzles(filepath: &Path, max_puzzles: Option<usize>) -> Result<Vec<String>, String> {
        let contenu = std::fs::read_to_string(filepath)
            .map_err(|e| format!("Impossible de lire {}: {}", filepath.display(), e))?;
        let mut puzzles: Vec<String> = Vec::new();
        for ligne in contenu.lines() {
            let l = ligne.trim();
            if l.chars().count() >= 81 {
                puzzles.push(l.chars().take(81).collect());
                if let Some(m) = max_puzzles {
                    if puzzles.len() >= m {
                        break;
                    }
                }
            }
        }
        if puzzles.is_empty() {
            return Err(format!("Aucun puzzle valide dans {}", filepath.display()));
        }
        Ok(puzzles)
    }

    /// Probleme de Satisfaction de Contraintes (CSP) binaire.
    struct Csp {
        variables: Vec<(usize, usize)>,
        domains: Vec<Vec<u8>>,
        neighbors: Vec<Vec<usize>>,
        num_assignments: u64,
        num_backtracks: u64,
    }

    impl Csp {
        fn copy_domains(&self) -> Vec<Vec<u8>> {
            self.domains.clone()
        }

        fn get_arcs(&self) -> Vec<(usize, usize)> {
            let mut arcs = Vec::new();
            for var in 0..self.variables.len() {
                for &voisin in &self.neighbors[var] {
                    arcs.push((var, voisin));
                }
            }
            arcs
        }
    }

    /// Construit le CSP necessaire a l'exercice.
    fn build_csp(grid: &SudokuGrid) -> Csp {
        let mut variables = Vec::new();
        let mut domains = vec![Vec::new(); 81];
        let mut neighbors = vec![Vec::new(); 81];

        for r in 0..9 {
            for c in 0..9 {
                let var = r * 9 + c;
                variables.push((r, c));
                domains[var] = if grid.cells[r][c] == 0 {
                    (1u8..=9).collect()
                } else {
                    vec![grid.cells[r][c]]
                };

                let mut neighbor_set = BTreeSet::new();
                for k in 0..9 {
                    if k != c {
                        neighbor_set.insert(r * 9 + k);
                    }
                    if k != r {
                        neighbor_set.insert(k * 9 + c);
                    }
                }
                let block_row = (r / 3) * 3;
                let block_col = (c / 3) * 3;
                for br in block_row..(block_row + 3) {
                    for bc in block_col..(block_col + 3) {
                        if br != r || bc != c {
                            neighbor_set.insert(br * 9 + bc);
                        }
                    }
                }
                neighbors[var] = neighbor_set.into_iter().collect();
            }
        }

        Csp {
            variables,
            domains,
            neighbors,
            num_assignments: 0,
            num_backtracks: 0,
        }
    }

    /// Infrastructure AC-3 fournie a l'etudiant.
    fn ac3_revise(
        csp: &Csp,
        xi: usize,
        xj: usize,
        current_domains: &mut [Vec<u8>],
    ) -> bool {
        let mut revised = false;
        let mut to_remove = Vec::new();
        for &val_i in &current_domains[xi] {
            let has_support = current_domains[xj]
                .iter()
                .any(|&val_j| val_i != val_j);
            if !has_support {
                to_remove.push(val_i);
                revised = true;
            }
        }
        for valeur in to_remove {
            current_domains[xi].retain(|&x| x != valeur);
        }
        revised
    }

    fn ac3_run(
        csp: &Csp,
        current_domains: &mut [Vec<u8>],
        arcs: Option<Vec<(usize, usize)>>,
    ) -> bool {
        let arcs_initiaux = match arcs {
            Some(arcs) if !arcs.is_empty() => arcs,
            _ => csp.get_arcs(),
        };
        let mut queue: VecDeque<(usize, usize)> = arcs_initiaux.into_iter().collect();
        while let Some((xi, xj)) = queue.pop_front() {
            if ac3_revise(csp, xi, xj, current_domains) {
                if current_domains[xi].is_empty() {
                    return false;
                }
                for &xk in &csp.neighbors[xi] {
                    if xk != xj {
                        queue.push_back((xk, xi));
                    }
                }
            }
        }
        true
    }

    /// Applique AC-3 en pre-processing et affiche les statistiques de reduction.
    ///
    /// grid : Grille de Sudoku a analyser
    fn preprocess_ac3(_grid: &SudokuGrid) {
        // TODO etudiant : implémenter le pre-processing AC-3
        // Étape 1 : construire le CSP avec SudokuCSPBuilder.build_csp
        // Étape 2 : copier les domaines avec csp.copy_domains()
        // Étape 3 : compter les valeurs initiales
        // Étape 4 : appliquer AC3.run(csp, current_domains)
        // Étape 5 : compter les valeurs restantes et afficher la reduction
    }

    pub fn run() {
        // Test sur la grille de référence
        let notebook_dir = std::env::current_dir().unwrap_or_else(|_| std::path::PathBuf::from("."));
        let puzzles_dir = notebook_dir.join("Puzzles");
        let resultat: Result<SudokuGrid, String> = (|| {
            let easy_puzzles = load_puzzles(&puzzles_dir.join("Sudoku_Easy51.txt"), Some(5))?;
            let premiere = easy_puzzles
                .first()
                .ok_or_else(|| "Le corpus ne contient aucun puzzle".to_string())?;
            SudokuGrid::from_string(premiere)
        })();

        match resultat {
            Ok(grid_pp) => {
                preprocess_ac3(&grid_pp);
                println!("Exercice a completer");
            }
            Err(erreur) => {
                eprintln!("Erreur de chargement du corpus: {}", erreur);
            }
        }
    }
}

exercise_27::run();


Exercice a completer


## 9. MAC (Maintaining Arc Consistency)

L'algorithme **MAC** combine le backtracking avec AC-3 : après chaque assignation, on maintient la consistance d'arc sur tout le CSP.

La mécanique reprend le solveur du forward checking, mais remplace la propagation à un niveau par un **point fixe AC-3** : au lieu de simplement retirer la valeur posée chez les voisins, MAC exécute AC-3 sur la mini-file des arcs (voisin, variable) concernés par l'assignation — et si un retrait vide à son tour le domaine d'une troisième variable, l'élimination continue de se propager jusqu'au point fixe. Deux gestes techniques rendent le retour arrière possible : le domaine de la variable assignée est **réduit à {val}** avant de lancer AC-3 (l'assignation devient une contrainte comme une autre), et l'état complet des domaines est **sauvegardé** (`saved_domains`) puis restauré à l'échec — chaque niveau de récursion travaille sur sa copie.

L'exercice de la section 8 (pré-traitement AC-3 seul, sans recherche) mesurera ce que cette élimination sait déjà faire sur une grille entière ; MAC, lui, relance la même machinerie à chaque nœud de la recherche. À ce prix, chaque assignation hérite des conséquences de toutes les précédentes — et la question du benchmark devient : ce surcoût par nœud est-il remboursé par les branches mortes évitées ?

In [15]:
// fn solve (MAC)
mod mac_29 {
    use std::cmp::Reverse;
    use std::collections::{BTreeMap, VecDeque};

    /// Probleme de Satisfaction de Contraintes (CSP) binaire.
    struct Csp {
        variables: Vec<(usize, usize)>,
        domains: Vec<Vec<u8>>,
        neighbors: Vec<Vec<usize>>,
        num_assignments: u64,
        num_backtracks: u64,
    }

    impl Csp {
        fn is_consistent(&self, var: usize, val: u8, assignment: &BTreeMap<usize, u8>) -> bool {
            for &voisin in &self.neighbors[var] {
                if let Some(&valeur_voisin) = assignment.get(&voisin) {
                    if val == valeur_voisin {
                        return false;
                    }
                }
            }
            true
        }

        fn is_complete(&self, assignment: &BTreeMap<usize, u8>) -> bool {
            assignment.len() == self.variables.len()
        }

        fn copy_domains(&self) -> Vec<Vec<u8>> {
            self.domains.clone()
        }

        fn get_arcs(&self) -> Vec<(usize, usize)> {
            let mut arcs = Vec::new();
            for var in 0..self.variables.len() {
                for &voisin in &self.neighbors[var] {
                    arcs.push((var, voisin));
                }
            }
            arcs
        }
    }

    fn select_mrv(csp: &Csp, assignment: &BTreeMap<usize, u8>, current_domains: &[Vec<u8>]) -> usize {
        let mut meilleure: Option<(usize, Reverse<usize>, usize)> = None;
        for &(r, c) in &csp.variables {
            let v = r * 9 + c;
            if assignment.contains_key(&v) { continue; }
            let remaining = current_domains[v]
                .iter()
                .filter(|&&val| csp.is_consistent(v, val, assignment))
                .count();
            let degree = csp.neighbors[v]
                .iter()
                .filter(|n| !assignment.contains_key(n))
                .count();
            let candidat = (remaining, Reverse(degree), v);
            match meilleure {
                None => meilleure = Some(candidat),
                Some(ref m) if candidat < *m => meilleure = Some(candidat),
                _ => {}
            }
        }
        meilleure.map(|(_, _, v)| v).unwrap_or(0)
    }

    fn order_lcv(csp: &Csp, var: usize, assignment: &BTreeMap<usize, u8>, current_domains: &[Vec<u8>]) -> Vec<u8> {
        let mut paires: Vec<(u8, usize)> = Vec::new();
        for &val in &current_domains[var] {
            let mut count = 0usize;
            for &voisin in &csp.neighbors[var] {
                if !assignment.contains_key(&voisin) {
                    for &nval in &current_domains[voisin] {
                        if val == nval { count += 1; }
                    }
                }
            }
            paires.push((val, count));
        }
        paires.sort_by_key(|&(_, c)| c);
        paires.into_iter().map(|(v, _)| v).collect()
    }

    fn ac3_revise(csp: &Csp, xi: usize, xj: usize, current_domains: &mut [Vec<u8>]) -> bool {
        let mut revised = false;
        let mut to_remove: Vec<u8> = Vec::new();
        for &val_i in &current_domains[xi] {
            let has_support = current_domains[xj].iter().any(|&val_j| val_i != val_j);
            if !has_support {
                to_remove.push(val_i);
                revised = true;
            }
        }
        for val in to_remove {
            current_domains[xi].retain(|&x| x != val);
        }
        revised
    }

    fn ac3_run(csp: &Csp, current_domains: &mut [Vec<u8>], arcs: Option<Vec<(usize, usize)>>) -> bool {
        let arcs_initiaux = match arcs {
            Some(arcs) if !arcs.is_empty() => arcs,
            _ => csp.get_arcs(),
        };
        let mut queue: VecDeque<(usize, usize)> = arcs_initiaux.into_iter().collect();
        while let Some((xi, xj)) = queue.pop_front() {
            if ac3_revise(csp, xi, xj, current_domains) {
                if current_domains[xi].is_empty() { return false; }
                for &xk in &csp.neighbors[xi] {
                    if xk != xj {
                        queue.push_back((xk, xi));
                    }
                }
            }
        }
        true
    }

    /// MAC (Maintaining Arc Consistency) : Backtracking + AC-3.
    fn mac_solve(
        csp: &mut Csp,
        assignment: &mut BTreeMap<usize, u8>,
        current_domains: &mut Vec<Vec<u8>>,
    ) -> bool {
        if csp.is_complete(assignment) { return true; }

        let var = select_mrv(csp, assignment, current_domains);
        let valeurs = order_lcv(csp, var, assignment, current_domains);

        for val in valeurs {
            csp.num_assignments += 1;
            if csp.is_consistent(var, val, assignment) {
                assignment.insert(var, val);
                let saved_domains = current_domains.clone();
                current_domains[var] = vec![val];
                let arcs: Vec<(usize, usize)> = csp.neighbors[var]
                    .iter()
                    .copied()
                    .filter(|&n| !assignment.contains_key(&n))
                    .map(|n| (n, var))
                    .collect();
                let success = ac3_run(csp, current_domains, Some(arcs));

                if success && mac_solve(csp, assignment, current_domains) {
                    return true;
                }

                *current_domains = saved_domains;
                assignment.remove(&var);
                csp.num_backtracks += 1;
            }
        }

        false
    }

    pub fn run() {
        println!("MAC defini.");
    }
}

mac_29::run();


MAC defini.


## 10. Test et Benchmark

La grille témoin (le premier puzzle facile, 45 indices) sert d'abord de banc unique à MAC, puis le benchmark compare les quatre stratégies de backtracking sur deux puzzles faciles, et FC contre MAC sur quatre grilles du corpus difficile. Trois métriques sont relevées à chaque ligne : les **assignations** (combien de fois le solveur a tenté de poser une valeur), les **backtracks** (combien de fois il a dû défaire) et le **temps**. Les deux premières sont des compteurs d'opérations, donc comparables d'une machine et d'un langage à l'autre ; le temps dépend du matériel et ne se lit qu'en ordre de grandeur.

La colonne de succès compte les grilles résolues **et re-validées** par `is_valid()` à l'issue de la recherche : un solveur qui remplirait la grille en violant une contrainte serait compté pour un échec, pas pour une réussite. Toutes les stratégies partent de la même instance CSP et du même corpus — seuls l'ordre d'attaque et la propagation changent.

In [16]:
// Test sur un puzzle (MAC)
mod mac_test_31 {
    use std::cmp::Reverse;
    use std::collections::{BTreeMap, VecDeque};
    use std::fmt;
    use std::path::Path;
    use std::time::Instant;

    /// Representation d'une grille de Sudoku 9x9.
    #[derive(Clone)]
    struct SudokuGrid {
        cells: [[u8; 9]; 9],
    }

    impl SudokuGrid {
        fn from_string(s: &str) -> Result<Self, String> {
            let nettoye: String = s
                .chars()
                .map(|c| if c == '.' { '0' } else { c })
                .filter(|c| *c != ' ' && *c != '\n' && *c != '\r')
                .collect();
            if nettoye.chars().count() != 81 {
                return Err("La chaine doit avoir 81 caracteres".to_string());
            }
            let mut grille = SudokuGrid { cells: [[0u8; 9]; 9] };
            for (i, c) in nettoye.chars().enumerate() {
                let v: u8 = c
                    .to_digit(10)
                    .ok_or_else(|| "La chaine doit contenir uniquement des chiffres".to_string())? as u8;
                grille.cells[i / 9][i % 9] = v;
            }
            Ok(grille)
        }

        fn is_valid(&self) -> bool {
            // Lignes
            for r in 0..9 {
                let mut vues = [false; 10];
                for c in 0..9 {
                    let valeur = self.cells[r][c];
                    if valeur != 0 {
                        let index = valeur as usize;
                        if vues[index] {
                            return false;
                        }
                        vues[index] = true;
                    }
                }
            }

            // Colonnes
            for c in 0..9 {
                let mut vues = [false; 10];
                for r in 0..9 {
                    let valeur = self.cells[r][c];
                    if valeur != 0 {
                        let index = valeur as usize;
                        if vues[index] {
                            return false;
                        }
                        vues[index] = true;
                    }
                }
            }

            // Blocs 3x3
            for block_row in 0..3 {
                for block_col in 0..3 {
                    let mut vues = [false; 10];
                    for r in (block_row * 3)..(block_row * 3 + 3) {
                        for c in (block_col * 3)..(block_col * 3 + 3) {
                            let valeur = self.cells[r][c];
                            if valeur != 0 {
                                let index = valeur as usize;
                                if vues[index] {
                                    return false;
                                }
                                vues[index] = true;
                            }
                        }
                    }
                }
            }

            true
        }
    }

    impl fmt::Display for SudokuGrid {
        fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
            let mut lignes: Vec<String> = Vec::new();
            for r in 0..9 {
                if r > 0 && r % 3 == 0 {
                    lignes.push("-".repeat(21));
                }
                let mut row_str = String::new();
                for c in 0..9 {
                    if c > 0 && c % 3 == 0 {
                        row_str.push_str("| ");
                    }
                    let val = self.cells[r][c];
                    if val != 0 {
                        row_str.push_str(&val.to_string());
                    } else {
                        row_str.push('.');
                    }
                    row_str.push(' ');
                }
                lignes.push(row_str);
            }
            write!(f, "{}", lignes.join("\n"))
        }
    }

    fn load_puzzles(filepath: &Path, max_puzzles: Option<usize>) -> Result<Vec<String>, String> {
        let contenu = std::fs::read_to_string(filepath)
            .map_err(|e| format!("Impossible de lire {}: {}", filepath.display(), e))?;
        let mut puzzles: Vec<String> = Vec::new();
        for ligne in contenu.lines() {
            let l = ligne.trim();
            if l.chars().count() >= 81 {
                puzzles.push(l.chars().take(81).collect());
                if let Some(m) = max_puzzles {
                    if puzzles.len() >= m {
                        break;
                    }
                }
            }
        }
        if puzzles.is_empty() {
            return Err(format!("Aucun puzzle valide dans {}", filepath.display()));
        }
        Ok(puzzles)
    }

    struct Csp {
        variables: Vec<(usize, usize)>,
        domains: Vec<Vec<u8>>,
        neighbors: Vec<Vec<usize>>,
        num_assignments: u64,
        num_backtracks: u64,
    }

    impl Csp {
        fn is_consistent(&self, var: usize, val: u8, assignment: &BTreeMap<usize, u8>) -> bool {
            for &voisin in &self.neighbors[var] {
                if let Some(&valeur_voisin) = assignment.get(&voisin) {
                    if val == valeur_voisin {
                        return false;
                    }
                }
            }
            true
        }

        fn is_complete(&self, assignment: &BTreeMap<usize, u8>) -> bool {
            assignment.len() == self.variables.len()
        }

        fn get_arcs(&self) -> Vec<(usize, usize)> {
            let mut arcs = Vec::new();
            for var in 0..self.variables.len() {
                for &voisin in &self.neighbors[var] {
                    arcs.push((var, voisin));
                }
            }
            arcs
        }
    }

    fn build_csp(grid: &SudokuGrid) -> Csp {
        let mut variables = Vec::new();
        let mut domains = vec![Vec::new(); 81];
        let mut neighbors = vec![Vec::new(); 81];
        for i in 0..9 {
            for j in 0..9 {
                variables.push((i, j));
                let v = i * 9 + j;
                domains[v] = if grid.cells[i][j] == 0 {
                    (1u8..=9).collect()
                } else {
                    vec![grid.cells[i][j]]
                };
                let mut neighbor_set: std::collections::BTreeSet<usize> = std::collections::BTreeSet::new();
                for k in 0..9 {
                    if k != j {
                        neighbor_set.insert(i * 9 + k);
                    }
                    if k != i {
                        neighbor_set.insert(k * 9 + j);
                    }
                }
                let br = (i / 3) * 3;
                let bc = (j / 3) * 3;
                for r in br..(br + 3) {
                    for c in bc..(bc + 3) {
                        if r != i || c != j {
                            neighbor_set.insert(r * 9 + c);
                        }
                    }
                }
                neighbors[v] = neighbor_set.into_iter().collect();
            }
        }
        Csp {
            variables,
            domains,
            neighbors,
            num_assignments: 0,
            num_backtracks: 0,
        }
    }

    fn select_mrv(csp: &Csp, assignment: &BTreeMap<usize, u8>, domains: &[Vec<u8>]) -> usize {
        let mut best: Option<(usize, Reverse<usize>, usize)> = None;
        for &(r, c) in &csp.variables {
            let v = r * 9 + c;
            if assignment.contains_key(&v) {
                continue;
            }

            let mut remaining = 0usize;
            for &valeur in &domains[v] {
                if csp.is_consistent(v, valeur, assignment) {
                    remaining += 1;
                }
            }

            let mut degree = 0usize;
            for &voisin in &csp.neighbors[v] {
                if !assignment.contains_key(&voisin) {
                    degree += 1;
                }
            }

            let candidate = (remaining, Reverse(degree), v);
            if best.as_ref().map_or(true, |b| candidate < *b) {
                best = Some(candidate);
            }
        }
        best.map(|(_, _, v)| v).unwrap_or(0)
    }

    fn order_lcv(csp: &Csp, var: usize, assignment: &BTreeMap<usize, u8>, domains: &[Vec<u8>]) -> Vec<u8> {
        let mut pairs: Vec<(u8, usize)> = Vec::new();
        for &val in &domains[var] {
            let mut conflicts = 0usize;
            for &voisin in &csp.neighbors[var] {
                if !assignment.contains_key(&voisin) {
                    for &nval in &domains[voisin] {
                        if val == nval {
                            conflicts += 1;
                        }
                    }
                }
            }
            pairs.push((val, conflicts));
        }
        pairs.sort_by_key(|&(_, c)| c);
        pairs.into_iter().map(|(v, _)| v).collect()
    }

    fn ac3_revise(csp: &Csp, xi: usize, xj: usize, domains: &mut [Vec<u8>]) -> bool {
        let mut revised = false;
        let mut to_remove = Vec::new();
        for &val_i in &domains[xi] {
            let has_support = domains[xj]
                .iter()
                .any(|&val_j| val_i != val_j);
            if !has_support {
                to_remove.push(val_i);
                revised = true;
            }
        }
        for val in to_remove {
            domains[xi].retain(|&x| x != val);
        }
        revised
    }

    fn ac3_run(csp: &Csp, domains: &mut [Vec<u8>], arcs: Option<Vec<(usize, usize)>>) -> bool {
        let arcs_initiaux = match arcs {
            Some(arcs) if !arcs.is_empty() => arcs,
            _ => csp.get_arcs(),
        };
        let mut queue: VecDeque<(usize, usize)> = arcs_initiaux.into_iter().collect();
        while let Some((xi, xj)) = queue.pop_front() {
            if ac3_revise(csp, xi, xj, domains) {
                if domains[xi].is_empty() {
                    return false;
                }
                for &xk in &csp.neighbors[xi] {
                    if xk != xj {
                        queue.push_back((xk, xi));
                    }
                }
            }
        }
        true
    }

    fn mac_solve(csp: &mut Csp, assignment: &mut BTreeMap<usize, u8>, domains: &mut Vec<Vec<u8>>) -> bool {
        if csp.is_complete(assignment) {
            return true;
        }
        let var = select_mrv(csp, assignment, domains);
        for val in order_lcv(csp, var, assignment, domains) {
            csp.num_assignments += 1;
            if csp.is_consistent(var, val, assignment) {
                assignment.insert(var, val);
                let saved = domains.clone();
                domains[var] = vec![val];
                let arcs: Vec<(usize, usize)> = csp.neighbors[var]
                    .iter()
                    .copied()
                    .filter(|&n| !assignment.contains_key(&n))
                    .map(|n| (n, var))
                    .collect();
                let success = ac3_run(csp, domains, Some(arcs));
                if success && mac_solve(csp, assignment, domains) {
                    return true;
                }
                *domains = saved;
                assignment.remove(&var);
                csp.num_backtracks += 1;
            }
        }
        false
    }

    pub fn run() {
        let notebook_dir = std::env::current_dir().unwrap_or_else(|_| std::path::PathBuf::from("."));
        let puzzles_dir = notebook_dir.join("Puzzles");
        let resultat: Result<SudokuGrid, String> = (|| {
            let easy_puzzles = load_puzzles(&puzzles_dir.join("Sudoku_Easy51.txt"), Some(5))?;
            let premiere = easy_puzzles
                .first()
                .ok_or_else(|| "Le corpus ne contient aucun puzzle".to_string())?;
            SudokuGrid::from_string(premiere)
        })();

        let puzzle = match resultat {
            Ok(puzzle) => puzzle,
            Err(erreur) => {
                eprintln!("Erreur de chargement du corpus: {}", erreur);
                return;
            }
        };

        println!("Puzzle original:");
        println!("{}", puzzle);

        let mut csp = build_csp(&puzzle);
        let start = Instant::now();
        let mut assignment = BTreeMap::new();
        let mut domains = csp.domains.clone();
        let found = mac_solve(&mut csp, &mut assignment, &mut domains);
        let elapsed = start.elapsed().as_secs_f64();

        if found {
            let mut result = puzzle.clone();
            for (&v, &value) in assignment.iter() {
                result.cells[v / 9][v % 9] = value;
            }
            println!("\nSolution (MAC): {:.1}ms, {} assignations, {} backtracks", elapsed * 1000.0, csp.num_assignments, csp.num_backtracks);
            println!("{}", result);
            println!("\nSolution valide: {}", if result.is_valid() { "True" } else { "False" });
        } else {
            println!("Pas de solution trouvee");
        }
    }
}

mac_test_31::run();


Puzzle original:


9 . 2 | . . 5 | 4 . 3 


1 . . | . 6 3 | . 2 5 


5 . 8 | 4 . 7 | . 6 . 


---------------------


. 2 6 | 3 . 9 | . . 1 


... (15 outputs left out)

2 4 9 | 5 3 8 | 6 1 7 


7 1 5 | 2 9 6 | 3 8 4 


6 8 3 | 7 4 1 | 9 5 2 


Solution valide: True


### Lecture du résultat MAC : 81 assignations, 0 backtrack

Le triplet mesuré vaut une lecture ligne à ligne. **81 assignations** : exactement une par case — jamais le solveur n'a essayé une valeur qu'il a dû retirer. **0 backtrack** : aucun retour en arrière ; l'inférence (arc-consistance maintenue à chaque assignation) a éliminé les conflits avant même qu'ils ne surviennent. **Le temps affiché** est la durée mesurée par ce port Rust au moment de l'exécution : deux langages, deux implémentations, un même verdict structurel — sur une grille à 45 indices bien contrainte, MAC résout par pure propagation, la recherche ne fait qu'encaisser. Le temps se lit en ordre de grandeur et dépend du matériel : seule la paire 81/0 est stable, c'est elle qui porte la lecture. La ligne finale « Solution valide: True » referme la boucle : la solution est re-vérifiée indépendamment de l'algorithme qui l'a produite.

Rappel de la mécanique que ces chiffres traduisent — l'algorithme MAC :
1. Utilise MRV pour choisir la case la plus contrainte
2. Utilise LCV pour essayer les valeurs les moins contraignantes
3. Maintient la consistance d'arc après chaque assignation — c'est elle qui rend possibles 81 assignations sans un seul retour arrière


In [17]:
// Comparaison des strategies
// Les durees Rust sont mesurees a chaque execution ; elles ne sont pas figees.
mod strategies_33 {
    use std::cmp::Reverse;
    use std::collections::{BTreeMap, BTreeSet, VecDeque};
    use std::fmt;
    use std::path::Path;
    use std::time::Instant;

    #[derive(Clone)]
    struct SudokuGrid {
        cells: [[u8; 9]; 9],
    }

    impl SudokuGrid {
        fn from_string(s: &str) -> Result<Self, String> {
            let nettoye: String = s
                .chars()
                .map(|c| if c == '.' { '0' } else { c })
                .filter(|c| *c != ' ' && *c != '\n' && *c != '\r')
                .collect();
            if nettoye.chars().count() != 81 {
                return Err("La chaine doit avoir 81 caracteres".to_string());
            }
            let mut grille = SudokuGrid { cells: [[0u8; 9]; 9] };
            for (i, c) in nettoye.chars().enumerate() {
                let valeur: u8 = c
                    .to_digit(10)
                    .ok_or_else(|| "La chaine doit contenir uniquement des chiffres".to_string())?
                    as u8;
                grille.cells[i / 9][i % 9] = valeur;
            }
            Ok(grille)
        }

        fn is_valid(&self) -> bool {
            for r in 0..9 {
                let mut ligne = [false; 10];
                for c in 0..9 {
                    let valeur = self.cells[r][c];
                    if valeur != 0 {
                        if ligne[valeur as usize] {
                            return false;
                        }
                        ligne[valeur as usize] = true;
                    }
                }
            }
            for c in 0..9 {
                let mut colonne = [false; 10];
                for r in 0..9 {
                    let valeur = self.cells[r][c];
                    if valeur != 0 {
                        if colonne[valeur as usize] {
                            return false;
                        }
                        colonne[valeur as usize] = true;
                    }
                }
            }
            for bloc_r in 0..3 {
                for bloc_c in 0..3 {
                    let mut bloc = [false; 10];
                    for r in bloc_r * 3..bloc_r * 3 + 3 {
                        for c in bloc_c * 3..bloc_c * 3 + 3 {
                            let valeur = self.cells[r][c];
                            if valeur != 0 {
                                if bloc[valeur as usize] {
                                    return false;
                                }
                                bloc[valeur as usize] = true;
                            }
                        }
                    }
                }
            }
            true
        }
    }

    fn load_puzzles(chemin: &Path, maximum: Option<usize>) -> Result<Vec<String>, String> {
        let contenu = std::fs::read_to_string(chemin)
            .map_err(|e| format!("Impossible de lire {}: {}", chemin.display(), e))?;
        let mut puzzles = Vec::new();
        for ligne in contenu.lines() {
            let ligne = ligne.trim();
            if ligne.chars().count() >= 81 {
                puzzles.push(ligne.chars().take(81).collect());
                if let Some(maximum) = maximum {
                    if puzzles.len() >= maximum {
                        break;
                    }
                }
            }
        }
        if puzzles.is_empty() {
            return Err(format!("Aucun puzzle valide dans {}", chemin.display()));
        }
        Ok(puzzles)
    }

    struct Csp {
        variables: Vec<(usize, usize)>,
        domains: Vec<Vec<u8>>,
        neighbors: Vec<Vec<usize>>,
        num_assignments: u64,
        num_backtracks: u64,
    }

    impl Csp {
        fn is_consistent(&self, variable: usize, valeur: u8, assignment: &BTreeMap<usize, u8>) -> bool {
            self.neighbors[variable]
                .iter()
                .filter_map(|&voisin| assignment.get(&voisin))
                .all(|&valeur_voisin| valeur != valeur_voisin)
        }

        fn is_complete(&self, assignment: &BTreeMap<usize, u8>) -> bool {
            assignment.len() == self.variables.len()
        }

        fn copy_domains(&self) -> Vec<Vec<u8>> {
            self.domains.clone()
        }

        fn get_arcs(&self) -> Vec<(usize, usize)> {
            let mut arcs = Vec::new();
            for variable in 0..self.variables.len() {
                for &voisin in &self.neighbors[variable] {
                    arcs.push((variable, voisin));
                }
            }
            arcs
        }
    }

    fn build_csp(grid: &SudokuGrid) -> Csp {
        let mut variables = Vec::new();
        let mut domains = vec![Vec::new(); 81];
        let mut neighbors = vec![Vec::new(); 81];

        for r in 0..9 {
            for c in 0..9 {
                let variable = r * 9 + c;
                variables.push((r, c));
                domains[variable] = if grid.cells[r][c] == 0 {
                    (1u8..=9).collect()
                } else {
                    vec![grid.cells[r][c]]
                };

                let mut neighbor_set = BTreeSet::new();
                for k in 0..9 {
                    if k != c {
                        neighbor_set.insert(r * 9 + k);
                    }
                    if k != r {
                        neighbor_set.insert(k * 9 + c);
                    }
                }
                let bloc_r = (r / 3) * 3;
                let bloc_c = (c / 3) * 3;
                for nr in bloc_r..bloc_r + 3 {
                    for nc in bloc_c..bloc_c + 3 {
                        if nr != r || nc != c {
                            neighbor_set.insert(nr * 9 + nc);
                        }
                    }
                }
                neighbors[variable] = neighbor_set.into_iter().collect();
            }
        }

        Csp {
            variables,
            domains,
            neighbors,
            num_assignments: 0,
            num_backtracks: 0,
        }
    }

    fn apply_solution(grid: &mut SudokuGrid, assignment: &BTreeMap<usize, u8>) {
        for (&variable, &valeur) in assignment.iter() {
            grid.cells[variable / 9][variable % 9] = valeur;
        }
    }

    fn select_mrv(
        csp: &Csp,
        assignment: &BTreeMap<usize, u8>,
        current_domains: &[Vec<u8>],
    ) -> usize {
        let mut meilleure: Option<(usize, Reverse<usize>, usize)> = None;
        for &(r, c) in &csp.variables {
            let variable = r * 9 + c;
            if assignment.contains_key(&variable) {
                continue;
            }
            let remaining = current_domains[variable]
                .iter()
                .filter(|&&valeur| csp.is_consistent(variable, valeur, assignment))
                .count();
            let degree = csp.neighbors[variable]
                .iter()
                .filter(|voisin| !assignment.contains_key(voisin))
                .count();
            let candidat = (remaining, Reverse(degree), variable);
            if meilleure.as_ref().map_or(true, |actuelle| candidat < *actuelle) {
                meilleure = Some(candidat);
            }
        }
        meilleure.map(|(_, _, variable)| variable).unwrap_or(0)
    }

    fn order_lcv(
        csp: &Csp,
        variable: usize,
        assignment: &BTreeMap<usize, u8>,
        current_domains: &[Vec<u8>],
    ) -> Vec<u8> {
        let mut valeurs = Vec::new();
        for &valeur in &current_domains[variable] {
            let mut conflits = 0usize;
            for &voisin in &csp.neighbors[variable] {
                if !assignment.contains_key(&voisin) {
                    for &valeur_voisine in &current_domains[voisin] {
                        if valeur == valeur_voisine {
                            conflits += 1;
                        }
                    }
                }
            }
            valeurs.push((valeur, conflits));
        }
        valeurs.sort_by_key(|&(_, conflits)| conflits);
        valeurs.into_iter().map(|(valeur, _)| valeur).collect()
    }

    fn backtracking_simple(
        csp: &mut Csp,
        assignment: &mut BTreeMap<usize, u8>,
    ) -> bool {
        if csp.is_complete(assignment) {
            return true;
        }
        let variable = csp
            .variables
            .iter()
            .map(|&(r, c)| r * 9 + c)
            .find(|variable| !assignment.contains_key(variable))
            .unwrap();

        for valeur in csp.domains[variable].clone() {
            csp.num_assignments += 1;
            if csp.is_consistent(variable, valeur, assignment) {
                assignment.insert(variable, valeur);
                if backtracking_simple(csp, assignment) {
                    return true;
                }
                assignment.remove(&variable);
                csp.num_backtracks += 1;
            }
        }
        false
    }

    fn backtracking_improved(
        csp: &mut Csp,
        assignment: &mut BTreeMap<usize, u8>,
        current_domains: &mut Vec<Vec<u8>>,
        use_mrv: bool,
        use_lcv: bool,
    ) -> bool {
        if csp.is_complete(assignment) {
            return true;
        }
        let variable = if use_mrv {
            select_mrv(csp, assignment, current_domains)
        } else {
            csp.variables
                .iter()
                .map(|&(r, c)| r * 9 + c)
                .find(|variable| !assignment.contains_key(variable))
                .unwrap()
        };
        let valeurs = if use_lcv {
            order_lcv(csp, variable, assignment, current_domains)
        } else {
            current_domains[variable].clone()
        };

        for valeur in valeurs {
            csp.num_assignments += 1;
            if csp.is_consistent(variable, valeur, assignment) {
                assignment.insert(variable, valeur);
                if backtracking_improved(
                    csp,
                    assignment,
                    current_domains,
                    use_mrv,
                    use_lcv,
                ) {
                    return true;
                }
                assignment.remove(&variable);
                csp.num_backtracks += 1;
            }
        }
        false
    }

    fn fc_propagate(
        csp: &Csp,
        variable: usize,
        valeur: u8,
        assignment: &BTreeMap<usize, u8>,
        current_domains: &mut [Vec<u8>],
    ) -> (Vec<(usize, u8)>, bool) {
        let mut removals = Vec::new();
        for &voisin in &csp.neighbors[variable] {
            if !assignment.contains_key(&voisin) {
                let mut a_retirer = Vec::new();
                for &valeur_voisine in &current_domains[voisin] {
                    if valeur == valeur_voisine {
                        a_retirer.push(valeur_voisine);
                        removals.push((voisin, valeur_voisine));
                    }
                }
                for valeur_retiree in a_retirer {
                    current_domains[voisin].retain(|&x| x != valeur_retiree);
                }
                if current_domains[voisin].is_empty() {
                    return (removals, false);
                }
            }
        }
        (removals, true)
    }

    fn fc_restore(current_domains: &mut [Vec<u8>], removals: &[(usize, u8)]) {
        for &(variable, valeur) in removals {
            current_domains[variable].push(valeur);
        }
    }

    fn forward_checking(
        csp: &mut Csp,
        assignment: &mut BTreeMap<usize, u8>,
        current_domains: &mut Vec<Vec<u8>>,
    ) -> bool {
        if csp.is_complete(assignment) {
            return true;
        }
        let variable = select_mrv(csp, assignment, current_domains);
        for valeur in order_lcv(csp, variable, assignment, current_domains) {
            csp.num_assignments += 1;
            if csp.is_consistent(variable, valeur, assignment) {
                assignment.insert(variable, valeur);
                let (removals, success) =
                    fc_propagate(csp, variable, valeur, assignment, current_domains);
                if success && forward_checking(csp, assignment, current_domains) {
                    return true;
                }
                fc_restore(current_domains, &removals);
                assignment.remove(&variable);
                csp.num_backtracks += 1;
            }
        }
        false
    }

    fn ac3_revise(
        _csp: &Csp,
        xi: usize,
        xj: usize,
        current_domains: &mut [Vec<u8>],
    ) -> bool {
        let mut revised = false;
        let mut a_retirer = Vec::new();
        for &valeur_i in &current_domains[xi] {
            if !current_domains[xj]
                .iter()
                .any(|&valeur_j| valeur_i != valeur_j)
            {
                a_retirer.push(valeur_i);
                revised = true;
            }
        }
        for valeur in a_retirer {
            current_domains[xi].retain(|&x| x != valeur);
        }
        revised
    }

    fn ac3_run(
        csp: &Csp,
        current_domains: &mut [Vec<u8>],
        arcs: Option<Vec<(usize, usize)>>,
    ) -> bool {
        let arcs_initiaux = match arcs {
            Some(arcs) if !arcs.is_empty() => arcs,
            _ => csp.get_arcs(),
        };
        let mut queue: VecDeque<(usize, usize)> = arcs_initiaux.into_iter().collect();
        while let Some((xi, xj)) = queue.pop_front() {
            if ac3_revise(csp, xi, xj, current_domains) {
                if current_domains[xi].is_empty() {
                    return false;
                }
                for &xk in &csp.neighbors[xi] {
                    if xk != xj {
                        queue.push_back((xk, xi));
                    }
                }
            }
        }
        true
    }

    fn mac(
        csp: &mut Csp,
        assignment: &mut BTreeMap<usize, u8>,
        current_domains: &mut Vec<Vec<u8>>,
    ) -> bool {
        if csp.is_complete(assignment) {
            return true;
        }
        let variable = select_mrv(csp, assignment, current_domains);
        for valeur in order_lcv(csp, variable, assignment, current_domains) {
            csp.num_assignments += 1;
            if csp.is_consistent(variable, valeur, assignment) {
                assignment.insert(variable, valeur);
                let saved_domains = current_domains.clone();
                current_domains[variable] = vec![valeur];
                let arcs: Vec<(usize, usize)> = csp.neighbors[variable]
                    .iter()
                    .copied()
                    .filter(|&voisin| !assignment.contains_key(&voisin))
                    .map(|voisin| (voisin, variable))
                    .collect();
                let success = ac3_run(csp, current_domains, Some(arcs));
                if success && mac(csp, assignment, current_domains) {
                    return true;
                }
                *current_domains = saved_domains;
                assignment.remove(&variable);
                csp.num_backtracks += 1;
            }
        }
        false
    }

    #[derive(Clone, Copy)]
    enum CspStrategy {
        BacktrackingSimple,
        BacktrackingMrvLcv,
        ForwardChecking,
        Mac,
    }

    impl CspStrategy {
        fn nom(&self) -> &'static str {
            match self {
                Self::BacktrackingSimple => "BACKTRACKING SIMPLE",
                Self::BacktrackingMrvLcv => "BACKTRACKING MRV LCV",
                Self::ForwardChecking => "FORWARD CHECKING",
                Self::Mac => "MAC",
            }
        }
    }

    fn solve_with_strategy(
        grid: &SudokuGrid,
        strategy: CspStrategy,
    ) -> (Option<SudokuGrid>, u64, u64, f64) {
        let mut csp = build_csp(grid);
        let mut assignment = BTreeMap::new();
        let start = Instant::now();
        let found = match strategy {
            CspStrategy::BacktrackingSimple => backtracking_simple(&mut csp, &mut assignment),
            CspStrategy::BacktrackingMrvLcv => {
                let mut domains = csp.copy_domains();
                backtracking_improved(
                    &mut csp,
                    &mut assignment,
                    &mut domains,
                    true,
                    true,
                )
            }
            CspStrategy::ForwardChecking => {
                let mut domains = csp.copy_domains();
                forward_checking(&mut csp, &mut assignment, &mut domains)
            }
            CspStrategy::Mac => {
                let mut domains = csp.copy_domains();
                mac(&mut csp, &mut assignment, &mut domains)
            }
        };
        let elapsed = start.elapsed().as_secs_f64();
        if found {
            let mut result = grid.clone();
            apply_solution(&mut result, &assignment);
            (Some(result), csp.num_assignments, csp.num_backtracks, elapsed)
        } else {
            (None, csp.num_assignments, csp.num_backtracks, elapsed)
        }
    }

    pub fn run() -> Result<(), String> {
        let notebook_dir = std::env::current_dir().unwrap_or_else(|_| std::path::PathBuf::from("."));
        let puzzles_dir = notebook_dir.join("Puzzles");
        let easy_puzzles = load_puzzles(&puzzles_dir.join("Sudoku_Easy51.txt"), Some(2))?;
        if easy_puzzles.len() < 2 {
            return Err("Le corpus facile doit contenir au moins 2 puzzles".to_string());
        }
        let hard_puzzles = load_puzzles(&puzzles_dir.join("Sudoku_top95.txt"), Some(4))?;
        if hard_puzzles.len() < 4 {
            return Err("Le corpus top95 doit contenir au moins 4 puzzles".to_string());
        }

        let strategies = [
            CspStrategy::BacktrackingSimple,
            CspStrategy::BacktrackingMrvLcv,
            CspStrategy::ForwardChecking,
            CspStrategy::Mac,
        ];
        let num_benchmark_puzzles = 2usize;
        println!("\n=== Benchmark sur {} puzzles faciles ===", num_benchmark_puzzles);
        println!("{}", "=".repeat(80));
        println!(
            "{:<25} {:>10} {:>12} {:>12} {:>8}",
            "Strategie", "Assigns", "Backtracks", "Temps(ms)", "Succes"
        );
        println!("{}", "-".repeat(80));

        for strategy in strategies {
            let mut total_assigns = 0u64;
            let mut total_backtracks = 0u64;
            let mut total_time = 0.0f64;
            let mut successes = 0usize;
            for i in 0..num_benchmark_puzzles {
                let grid = SudokuGrid::from_string(&easy_puzzles[i])
                    .map_err(|e| format!("Puzzle facile {} invalide: {}", i, e))?;
                let (result, assigns, backtracks, elapsed) =
                    solve_with_strategy(&grid, strategy);
                total_assigns += assigns;
                total_backtracks += backtracks;
                total_time += elapsed;
                if let Some(solution) = result {
                    if solution.is_valid() {
                        successes += 1;
                    }
                }
            }
            println!(
                "{:<25} {:>10} {:>12} {:>12.0} {}/{}",
                strategy.nom(),
                total_assigns / num_benchmark_puzzles as u64,
                total_backtracks / num_benchmark_puzzles as u64,
                total_time * 1000.0,
                successes,
                num_benchmark_puzzles
            );
        }
        println!("{}", "=".repeat(80));

        let num_hard = hard_puzzles.len();
        println!();
        println!("=== FC vs MAC sur {} puzzles difficiles (top95) ===", num_hard);
        println!("{}", "=".repeat(70));
        println!(
            "{:<22} {:>10} {:>12} {:>12} {:>8}",
            "Strategie", "Assigns", "Backtracks", "Temps(ms)", "Succes"
        );
        println!("{}", "-".repeat(70));

        for strategy in [CspStrategy::ForwardChecking, CspStrategy::Mac] {
            let mut total_assigns = 0u64;
            let mut total_backtracks = 0u64;
            let mut total_time = 0.0f64;
            let mut successes = 0usize;
            for i in 0..num_hard {
                let grid = SudokuGrid::from_string(&hard_puzzles[i])
                    .map_err(|e| format!("Puzzle top95 {} invalide: {}", i, e))?;
                let (result, assigns, backtracks, elapsed) =
                    solve_with_strategy(&grid, strategy);
                total_assigns += assigns;
                total_backtracks += backtracks;
                total_time += elapsed;
                if let Some(solution) = result {
                    if solution.is_valid() {
                        successes += 1;
                    }
                }
            }
            println!(
                "{:<22} {:>10} {:>12} {:>12.0} {}/{}",
                strategy.nom(),
                total_assigns / num_hard as u64,
                total_backtracks / num_hard as u64,
                total_time * 1000.0,
                successes,
                num_hard
            );
        }
        println!("{}", "=".repeat(70));
        Ok(())
    }
}

if let Err(erreur) = strategies_33::run() {
    eprintln!("Erreur du benchmark: {}", erreur);
};

=== Benchmark sur 2 puzzles faciles ===


Strategie                    Assigns   Backtracks    Temps(ms)   Succes


--------------------------------------------------------------------------------


BACKTRACKING SIMPLE          2889322       499461         1292 2/2


BACKTRACKING MRV LCV             255            0           12 2/2


FORWARD CHECKING                  81            0            9 2/2


... (2 outputs left out)

Strategie                 Assigns   Backtracks    Temps(ms)   Succes


----------------------------------------------------------------------


FORWARD CHECKING             2143         2062          445 4/4


MAC                          1060          979          237 4/4


### Interprétation : Comparaison des stratégies

| Stratégie | Assignations | Backtracks | Analyse |
|-----------|-------------|------------|----------|
| **Simple** | Eleve | Eleve | Ne profite d'aucune optimisation |
| **MRV+LCV** | Réduit | Réduit | Fail-first + succeed-first |
| **Forward Checking** | Très réduit | Très réduit | Propagation 1-niveau |
| **MAC** | Minimal | Minimal | Propagation complète |

**Observations clés** :
1. **MRV** est l'heuristique la plus impactante (reduction souvent > 10x)
2. **FC** ajoute un gain significatif en détectant les échecs plus tot
3. **MAC** est optimal mais plus couteux par noeud (AC-3 à chaque pas)

> **Pourquoi FC et MAC donnent-ils des résultats identiques sur les puzzles faciles ?**
> Une grille facile (fortement contrainte) est déjà presque résolue par propagation unitaire : la propagation **1-niveau** de FC détecte à elle seule tous les culs-de-sac, et le fixpoint AC-3 de MAC n'y ajoute rien — les deux convergent vers 81 assignations et 0 backtrack. Leur écart théorique (profondeur de propagation) n'apparaît que sur des instances **faiblement contraintes**, où des domaines intermédiaires se vident en profondeur : c'est ce que mesure le benchmark **top95** ci-dessus, où FC accumule environ **deux fois plus de backtracks** que MAC. Le surcoût de MAC (AC-3 à chaque pas) ne se rentabilise donc que sur les instances où la propagation 1-niveau ne suffit pas.

#### Durées : valeurs commitées du carnet source contre mesures du port Rust

Les **compteurs** du tableau de la cellule 33 (assignations, backtracks, succès) sont des grandeurs d'opérations : ils se lisent à l'identique entre le jumeau Python et ce port Rust, et c'est cette lecture qui porte la démonstration. Les **durées**, en revanche, mesurent une machine *et* une implémentation : elles ne sont comparables qu'à l'intérieur d'une même exécution.

Le tableau ci-dessous met en regard, ligne par ligne, la durée **commitée** du carnet source (cellules 33, 38 et 31 du jumeau `Sudoku-06-AIMA-CSP-Python.ipynb`) et la durée **mesurée par ce port**, lue sur la ligne correspondante de la sortie exécutée des cellules 33, 38 et 31 de ce carnet. Le rapport est la durée Python divisée par la durée Rust : il vaut 1 lorsque les deux implémentations coûtent pareil, et croît avec l'avantage du port.

| Ligne du benchmark | Python commité (ms) | Rust mesuré (ms) | Rapport Python ÷ Rust | Encadrement dû à l'arrondi |
|---|---|---|---|---|
| BACKTRACKING SIMPLE — 2 puzzles faciles (cellule 33) | 9 672 | 1 292 | ≈ 7,5 | 7,48 – 7,49 |
| BACKTRACKING MRV LCV — 2 puzzles faciles (cellule 33) | 77 | 12 | ≈ 6,4 | 6,12 – 6,74 |
| FORWARD CHECKING — 2 puzzles faciles (cellule 33) | 51 | 9 | ≈ 5,7 | 5,32 – 6,06 |
| MAC — 2 puzzles faciles (cellule 33) | 90 | 11 | ≈ 8,2 | 7,78 – 8,62 |
| FORWARD CHECKING — 4 puzzles top95 (cellule 33) | 2 189 | 445 | ≈ 4,9 | 4,91 – 4,93 |
| MAC — 4 puzzles top95 (cellule 33) | 2 044 | 237 | ≈ 8,6 | 8,60 – 8,64 |
| BACKTRACKING MRV LCV — 3 puzzles (cellule 38) | 138 | 21 | ≈ 6,6 | 6,40 – 6,76 |
| CONFLICT BACK JUMPING — 3 puzzles (cellule 38) | 91 | 14 | 6,5 | 6,24 – 6,78 |
| FORWARD CHECKING — 3 puzzles (cellule 38) | 92 | 16 | ≈ 5,8 | 5,55 – 5,97 |
| MAC, résolution d'une grille (cellule 31) | 33,7 | 4,8 | ≈ 7,0 | 6,94 – 7,11 |

Sur ces dix lignes, le rapport le plus faible est celui de FC sur top95 et le plus fort celui de MAC sur top95 (colonne « Rapport » du tableau). La ligne la plus lourde est la première : le même parcours de **2 889 322 assignations et 499 461 retours** sur le même corpus prend 9 672 ms côté Python et 1 292 ms côté Rust, dont le rapport figure en première ligne du tableau. C'est ce rapport, et non une durée isolée, qui porte la comparaison.

**Arrondi et résolution** : les durées des cellules 33 et 38 s'affichent arrondies à la milliseconde des deux côtés (`{:.0}` dans ce port), celles de la cellule 31 au dixième de milliseconde. Une valeur arrondie s'écarte au plus d'une demi-unité de son dernier chiffre affiché ; la dernière colonne du tableau donne le rapport le plus petit et le plus grand compatibles avec cet écart sur les deux durées de la ligne. Les lignes courtes portent l'essentiel de l'incertitude : l'encadrement de FC sur les puzzles faciles est large, celui de la première ligne presque ponctuel. Aucune durée Rust de cette exécution ne s'affiche à 0 ms. Une durée arrondie à zéro ne donnerait pas un rapport infini : le rapport y serait **non mesurable à cette résolution**, et la ligne se lirait sur ses seuls compteurs.

**Précaution de lecture** : les valeurs Rust de ce tableau sont celles de l'exécution commitée de ce carnet ; une ré-exécution produit d'autres durées, et le tableau se relit alors contre la nouvelle sortie. Ces rapports dépendent des conditions d'exécution (matériel, charge machine, version de la toolchain Rust) ; les valeurs Python, elles, sont figées dans le bloc commité du carnet source, et rien ne garantit qu'elles aient été mesurées sur la même machine. Aucun de ces rapports n'est donc une mesure de performance portable — c'est la reproduction des **compteurs** qui est la preuve, la durée n'étant qu'un ordre de grandeur observé.

## Exemple cautionnaire : CBJ (Conflict-Based Backjumping)

### Principe

Le **CBJ** (Conflict-Based Backjumping) est un algorithme qui remonte directement au niveau responsable d'un conflit au lieu de faire du backtracking chronologique. En théorie, cette approche devrait accélérer la résolution en evitant d'explorer des branches inutiles.

**Attention** : comme nous allons le voir, CBJ illustré un piège classique en IA -- un algorithme theoriquement superieur qui ne l'est pas toujours en pratique. L'étude de ce cas vous aidera à comprendre pourquoi le choix d'un algorithme dépend fortement du problème.

L'implementation ci-dessous illustré les points clés de l'algorithme :

1. **Conflict set** : table de correspondance `conflict_sets[var]` (un `HashSet` de variables) des variables antérieures qui ont cause un conflit avec `var`
2. **Détection de conflit** : Quand `val` est inconsistant avec un voisin assigné `n`, ajouter `n` au conflict set de `var`
3. **Backjumping** : Quand `var` n'a plus de valeur viable, remonter directement à la variable la plus recente dans son conflict set
4. **Fusion** : Lors d'un backjump de Y vers X, fusionner `conflict_sets[Y] - {X}` dans `conflict_sets[X]`

### Pourquoi cet exemple est instructif

Sur les puzzles faciles, CBJ avec MRV fonctionne bien (peu d'assignations). Mais sur les puzzles **difficiles**, le backjumping seul ne compense pas l'absence de propagation de contraintes (Forward Checking, AC-3) : c'est une limite connue de CBJ. Le benchmark ci-dessous porte sur des puzzles faciles, ou CBJ converge ; il illustré que **le backjumping seul ne suffit pas** -- il faut le combiner avec des heuristiques de sélection de variable ET de propagation de contraintes.

**Leçon** : ne jugez pas un algorithme seulement sur des instances faciles. Un algorithme qui semble efficace sur des problèmes simples peut echouer sur des instances plus difficiles ou la propagation de contraintes est determinante.

In [18]:
// Conflict-Based Backjumping (CBJ)
mod conflict_backjumping_36 {
    use std::collections::{BTreeMap, BTreeSet, HashMap, HashSet};
    use std::path::Path;
    use std::time::Instant;

    #[derive(Clone)]
    struct SudokuGrid { cells: [[u8; 9]; 9] }

    impl SudokuGrid {
        fn from_string(s: &str) -> Result<Self, String> {
            let nettoye: String = s.chars().map(|c| if c == '.' { '0' } else { c })
                .filter(|c| *c != ' ' && *c != '\n' && *c != '\r').collect();
            if nettoye.chars().count() != 81 {
                return Err("La chaine doit avoir 81 caracteres".to_string());
            }
            let mut grille = SudokuGrid { cells: [[0u8; 9]; 9] };
            for (i, c) in nettoye.chars().enumerate() {
                let v: u8 = c.to_digit(10).ok_or_else(|| "La chaine doit avoir 81 caracteres".to_string())? as u8;
                grille.cells[i / 9][i % 9] = v;
            }
            Ok(grille)
        }

        fn is_valid(&self) -> bool {
            for r in 0..9 {
                let mut row = [false; 10];
                for c in 0..9 {
                    let v = self.cells[r][c];
                    if v != 0 {
                        if row[v as usize] { return false; }
                        row[v as usize] = true;
                    }
                }
            }
            for c in 0..9 {
                let mut col = [false; 10];
                for r in 0..9 {
                    let v = self.cells[r][c];
                    if v != 0 {
                        if col[v as usize] { return false; }
                        col[v as usize] = true;
                    }
                }
            }
            for br in 0..3 {
                for bc in 0..3 {
                    let mut block = [false; 10];
                    for r in br * 3..br * 3 + 3 {
                        for c in bc * 3..bc * 3 + 3 {
                            let v = self.cells[r][c];
                            if v != 0 {
                                if block[v as usize] { return false; }
                                block[v as usize] = true;
                            }
                        }
                    }
                }
            }
            true
        }

        fn is_complete(&self) -> bool {
            (0..9).all(|r| (0..9).all(|c| self.cells[r][c] != 0))
        }
    }

    fn load_puzzles(filepath: &Path, max_puzzles: Option<usize>) -> Result<Vec<String>, String> {
        let contenu = std::fs::read_to_string(filepath)
            .map_err(|e| format!("Impossible de lire {}: {}", filepath.display(), e))?;
        let mut puzzles = Vec::new();
        for ligne in contenu.lines() {
            let l = ligne.trim();
            if l.chars().count() >= 81 {
                puzzles.push(l.chars().take(81).collect());
                if let Some(m) = max_puzzles {
                    if puzzles.len() >= m { break; }
                }
            }
        }
        if puzzles.is_empty() {
            return Err(format!("Aucun puzzle valide dans {}", filepath.display()));
        }
        Ok(puzzles)
    }

    struct Csp {
        variables: Vec<(usize, usize)>,
        domains: Vec<Vec<u8>>,
        neighbors: Vec<Vec<usize>>,
        num_assignments: u64,
        num_backtracks: u64,
    }

    impl Csp {
        fn is_consistent(&self, var: usize, val: u8, assignment: &BTreeMap<usize, u8>) -> bool {
            self.neighbors[var].iter().filter_map(|&n| assignment.get(&n)).all(|&x| val != x)
        }
    }

    fn build_csp(grid: &SudokuGrid) -> Csp {
        let mut variables = Vec::new();
        let mut domains = vec![Vec::new(); 81];
        let mut neighbors = vec![Vec::new(); 81];
        for r in 0..9 {
            for c in 0..9 {
                let v = r * 9 + c;
                variables.push((r, c));
                domains[v] = if grid.cells[r][c] == 0 {
                    (1u8..=9).collect()
                } else {
                    vec![grid.cells[r][c]]
                };
                let mut ns = BTreeSet::new();
                for k in 0..9 {
                    if k != c { ns.insert(r * 9 + k); }
                    if k != r { ns.insert(k * 9 + c); }
                }
                let br = (r / 3) * 3;
                let bc = (c / 3) * 3;
                for nr in br..br + 3 {
                    for nc in bc..bc + 3 {
                        if nr != r || nc != c { ns.insert(nr * 9 + nc); }
                    }
                }
                neighbors[v] = ns.into_iter().collect();
            }
        }
        Csp { variables, domains, neighbors, num_assignments: 0, num_backtracks: 0 }
    }

    fn appliquer_solution(grid: &mut SudokuGrid, assignment: &BTreeMap<usize, u8>) {
        for (&v, &value) in assignment.iter() { grid.cells[v / 9][v % 9] = value; }
    }

    /// Conflict-Based Backjumping (CBJ) pedagogique, inspire de Prosser (1993).
    ///
    /// Version simplifiee combinant :
    /// - selection dynamique de variable (MRV) pour garantir la convergence,
    /// - collecte des conflict sets en descendant,
    /// - backjump vers la variable la plus recente du conflict set quand le
    ///   domaine courant est epuise (sinon, fallback chronologique).
    ///
    /// Contribution initiale : Evariste BALVAY (PR #378).
    struct ConflictBackjumping;

    impl ConflictBackjumping {
        fn select_var_mrv(csp: &Csp, assignment: &BTreeMap<usize, u8>) -> usize {
            let mut meilleure: Option<(usize, usize)> = None;
            for &(r, c) in &csp.variables {
                let v = r * 9 + c;
                if assignment.contains_key(&v) { continue; }
                let remaining = csp.domains[v].iter()
                    .filter(|&&val| csp.is_consistent(v, val, assignment)).count();
                if meilleure.map_or(true, |(m, _)| remaining < m) {
                    meilleure = Some((remaining, v));
                }
            }
            meilleure.map(|(_, v)| v).unwrap_or(0)
        }

        /// Nettoie l'assignation courante et transmet a la cible le conflict set
        /// de la variable quittee, en retirant la cible de ce set comme la source.
        fn transmettre_saut_non_chronologique(
            assignment: &mut BTreeMap<usize, u8>,
            conflict_sets: &mut HashMap<usize, HashSet<usize>>,
            var: usize,
            jump_target: usize,
        ) {
            // Ordre conforme a la source : l'assignation de la variable quittee
            // est retiree AVANT la transmission du conflict set a la cible.
            assignment.remove(&var);
            if let Some(conflits) = conflict_sets.get(&var).cloned() {
                conflict_sets.entry(jump_target).or_default().extend(
                    conflits.into_iter().filter(|&conflit| conflit != jump_target)
                );
            }
        }

        /// `Ok(assignment)` designe une solution ; `Err(None)` un echec sans
        /// backjump demande ; `Err(Some(var))` un backjump vers `var`.
        fn solve_cbj(
            csp: &mut Csp,
            assignment: &mut BTreeMap<usize, u8>,
            conflict_sets: &mut HashMap<usize, HashSet<usize>>,
            history: &mut Vec<usize>,
        ) -> Result<BTreeMap<usize, u8>, Option<usize>> {
            if assignment.len() == csp.variables.len() { return Ok(assignment.clone()); }

            let var = Self::select_var_mrv(csp, assignment);
            history.push(var);
            conflict_sets.insert(var, HashSet::new());

            for val in csp.domains[var].clone() {
                let mut rejecters = HashSet::new();
                for &voisin in &csp.neighbors[var] {
                    if let Some(&valeur) = assignment.get(&voisin) {
                        if val == valeur { rejecters.insert(voisin); }
                    }
                }
                if !rejecters.is_empty() {
                    conflict_sets.entry(var).or_default().extend(rejecters);
                    continue;
                }

                assignment.insert(var, val);
                csp.num_assignments += 1;
                match Self::solve_cbj(csp, assignment, conflict_sets, history) {
                    Ok(solution) => return Ok(solution),
                    Err(Some(jump)) if jump != var => {
                        // La source retire l'assignation courante avant de
                        // transmettre le saut, puis fusionne dans la cible le
                        // conflict set de la variable quittee, cible exclue.
                        Self::transmettre_saut_non_chronologique(
                            assignment,
                            conflict_sets,
                            var,
                            jump,
                        );
                        history.pop();
                        return Err(Some(jump));
                    }
                    Err(_) => {}
                }
                assignment.remove(&var);
            }

            csp.num_backtracks += 1;
            if let Some(conflits) = conflict_sets.get(&var) {
                if !conflits.is_empty() {
                    let mut cible = None;
                    let mut position_cible = 0usize;
                    for &conflit in conflits {
                        if let Some(position) = history.iter().position(|&v| v == conflit) {
                            if cible.is_none() || position > position_cible {
                                cible = Some(conflit);
                                position_cible = position;
                            }
                        }
                    }
                    if let Some(cible) = cible {
                        if position_cible < history.len() - 1 {
                            history.pop();
                            return Err(Some(cible));
                        }
                    }
                }
            }
            history.pop();
            Err(None)
        }

        fn solve(csp: &mut Csp) -> Option<BTreeMap<usize, u8>> {
            let mut assignment = BTreeMap::new();
            let mut conflict_sets = HashMap::new();
            let mut history = Vec::new();
            match Self::solve_cbj(csp, &mut assignment, &mut conflict_sets, &mut history) {
                Ok(solution) => Some(solution),
                Err(_) => None,
            }
        }
    }

    /// Test de regression du saut non chronologique : la variable quittee
    /// doit disparaitre de l'assignation et son conflict set doit etre fusionne dans
    /// la cible, sans y reinjecter la cible elle-meme.
    fn tester_saut_non_chronologique() {
        let mut assignment = BTreeMap::new();
        assignment.insert(1, 7);

        let mut conflict_sets = HashMap::new();
        let mut conflits_source = HashSet::new();
        conflits_source.insert(0);
        conflits_source.insert(2);
        conflict_sets.insert(1, conflits_source);
        let mut conflits_cible = HashSet::new();
        conflits_cible.insert(9);
        conflict_sets.insert(2, conflits_cible);

        ConflictBackjumping::transmettre_saut_non_chronologique(
            &mut assignment,
            &mut conflict_sets,
            1,
            2,
        );

        // Nettoyage de l'assignation avant transmission.
        assert!(!assignment.contains_key(&1));
        // Fusion du conflict set source dans le conflict set de la cible.
        assert!(conflict_sets.get(&2).unwrap().contains(&0));
        assert!(conflict_sets.get(&2).unwrap().contains(&9));
        // La cible ne doit etre reinjectee dans son propre conflict set.
        assert!(!conflict_sets.get(&2).unwrap().contains(&2));
    }

    pub fn run() -> Result<(), String> {
        tester_saut_non_chronologique();

        let notebook_dir = std::env::current_dir().unwrap_or_else(|_| std::path::PathBuf::from("."));
        let puzzles_dir = notebook_dir.join("Puzzles");
        let easy_puzzles = load_puzzles(&puzzles_dir.join("Sudoku_Easy51.txt"), Some(1))?;
        if easy_puzzles.is_empty() {
            return Err("Le corpus facile ne contient aucun puzzle".to_string());
        }
        let puzzle = SudokuGrid::from_string(&easy_puzzles[0])
            .map_err(|e| format!("Puzzle facile invalide: {}", e))?;
        let mut csp = build_csp(&puzzle);
        let start = Instant::now();
        let solution = ConflictBackjumping::solve(&mut csp);
        let elapsed = start.elapsed().as_secs_f64();

        if let Some(assignment) = solution {
            let mut resultat = puzzle.clone();
            appliquer_solution(&mut resultat, &assignment);
            let valide = resultat.is_valid() && resultat.is_complete();
            println!("CBJ (easy): {}ms, {} assigns, {} backtracks, valide={}", format!("{:.0}", elapsed * 1000.0), csp.num_assignments, csp.num_backtracks, if valide { "True" } else { "False" });
        }
        println!("ConflictBackjumping implemente.");
        Ok(())
    }
}

if let Err(erreur) = conflict_backjumping_36::run() {
    eprintln!("Erreur du test CBJ: {}", erreur);
};


CBJ (easy): 4ms, 81 assigns, 0 backtracks, valide=True


ConflictBackjumping implemente.


### Benchmark CBJ vs autres stratégies

L'implémentation du CBJ avec sélection MRV converge sur les puzzles faciles. Comparons maintenant ses performances avec le backtracking amélioré (MRV+LCV) et le Forward Checking sur plusieurs puzzles pour observer si le backjumping apporte un gain mesurable.

En lisant le tableau, gardez les deux colonnes séparées : les **assignations** mesurent le travail de descente, les **backtracks** le travail défait — un algorithme peut améliorer l'une sans toucher l'autre. C'est exactement le décor du « piège CBJ » annoncé en section d'exemple cautionnaire : les chiffres montrent un backjumping qui réduit le nombre de retours par rapport à MRV+LCV. Les durées mesurées dans le tableau restent locales : elles dépendent du matériel et de l'implémentation et ne sont pas directement comparables aux durées de la source Python. On peut donc formuler une hypothèse de coût : la gestion des ensembles de conflits et de l'historique peut absorber une partie du gain obtenu sur les retours en arrière. Cette lecture reste une hypothèse ; une exécution isolée ne démontre pas à elle seule cette causalité.

In [19]:
// Comparaison CBJ avec les autres strategies
mod strategies_cbj_38 {
    use std::cmp::Reverse;
    use std::collections::{BTreeMap, BTreeSet, HashSet};
    use std::path::Path;
    use std::time::Instant;

    #[derive(Clone)]
    struct SudokuGrid { cells: [[u8; 9]; 9] }

    impl SudokuGrid {
        fn from_string(s: &str) -> Result<Self, String> {
            let nettoye: String = s.chars().map(|c| if c == '.' { '0' } else { c }).filter(|c| *c != ' ' && *c != '\n' && *c != '\r').collect();
            if nettoye.chars().count() != 81 { return Err("La chaine doit avoir 81 caracteres".to_string()); }
            let mut grille = SudokuGrid { cells: [[0u8; 9]; 9] };
            for (i, c) in nettoye.chars().enumerate() { let v: u8 = c.to_digit(10).ok_or_else(|| "La chaine doit avoir 81 caracteres".to_string())? as u8; grille.cells[i / 9][i % 9] = v; }
            Ok(grille)
        }
        fn is_valid(&self) -> bool {
            for i in 0..9 {
                let mut row: Vec<u8> = self.cells[i].iter().copied().filter(|&v| v != 0).collect(); row.sort_unstable(); row.dedup();
                if row.len() != self.cells[i].iter().filter(|&&v| v != 0).count() { return false; }
                let mut col: Vec<u8> = (0..9).map(|r| self.cells[r][i]).filter(|&v| v != 0).collect(); col.sort_unstable(); col.dedup();
                if col.len() != (0..9).map(|r| self.cells[r][i]).filter(|&v| v != 0).count() { return false; }
            }
            for br in 0..3 { for bc in 0..3 { let mut b = Vec::new(); for r in (br*3)..(br*3+3) { for c in (bc*3)..(bc*3+3) { if self.cells[r][c] != 0 { b.push(self.cells[r][c]); } } } let n=b.len(); b.sort_unstable(); b.dedup(); if b.len()!=n { return false; } }}
            true
        }
    }

    fn load_puzzles(path: &Path, max: Option<usize>) -> Vec<String> {
        let data = std::fs::read_to_string(path).unwrap_or_default(); let mut out = Vec::new();
        for line in data.lines() { let s=line.trim(); if s.chars().count()>=81 { out.push(s.chars().take(81).collect()); if let Some(m)=max { if out.len()>=m { break; } } } } out
    }

    struct Csp { variables: Vec<(usize, usize)>, domains: Vec<Vec<u8>>, neighbors: Vec<Vec<usize>>, num_assignments: u64, num_backtracks: u64 }

    impl Csp {
        fn is_consistent(&self, var: usize, val: u8, assignment: &BTreeMap<usize, u8>) -> bool { self.neighbors[var].iter().filter_map(|&n| assignment.get(&n)).all(|&x| val != x) }
        fn is_complete(&self, assignment: &BTreeMap<usize, u8>) -> bool { assignment.len()==self.variables.len() }
        fn copy_domains(&self) -> Vec<Vec<u8>> { self.domains.clone() }
        fn get_arcs(&self) -> Vec<(usize, usize)> { let mut a=Vec::new(); for v in 0..self.variables.len() { for &n in &self.neighbors[v] { a.push((v,n)); } } a }
    }

    fn build_csp(grid: &SudokuGrid) -> Csp {
        let mut variables=Vec::new(); let mut domains=vec![Vec::new();81]; let mut neighbors=vec![Vec::new();81];
        for i in 0..9 { for j in 0..9 { variables.push((i,j)); let v=i*9+j; domains[v]=if grid.cells[i][j]==0 {(1u8..=9).collect()} else {vec![grid.cells[i][j]]}; let mut ns=BTreeSet::new(); for k in 0..9 { if k!=j {ns.insert(i*9+k);} if k!=i {ns.insert(k*9+j);} } let br=(i/3)*3; let bc=(j/3)*3; for r in br..(br+3) { for c in bc..(bc+3) { if r!=i||c!=j {ns.insert(r*9+c);} } } neighbors[v]=ns.into_iter().collect(); }}
        Csp{variables,domains,neighbors,num_assignments:0,num_backtracks:0}
    }

    fn appliquer(grid: &mut SudokuGrid, assignment: &BTreeMap<usize,u8>) { for (&v,&x) in assignment.iter() {grid.cells[v/9][v%9]=x;} }

    fn select_mrv(csp:&Csp, assignment:&BTreeMap<usize,u8>, domains:&[Vec<u8>])->usize {
        let mut best=None;
        for &(r,c) in &csp.variables { let v=r*9+c; if assignment.contains_key(&v){continue;} let n=domains[v].iter().filter(|&&x|csp.is_consistent(v,x,assignment)).count(); let d=csp.neighbors[v].iter().filter(|x|!assignment.contains_key(x)).count(); let cand=(n,Reverse(d),v); if best.as_ref().map_or(true,|b:&(usize,Reverse<usize>,usize)|cand<*b){best=Some(cand);} }
        best.map(|(_,_,v)|v).unwrap_or(0)
    }

    fn order_lcv(csp:&Csp,var:usize,assignment:&BTreeMap<usize,u8>,domains:&[Vec<u8>])->Vec<u8> {
        let mut pairs=Vec::new(); for &val in &domains[var] {let mut n=0; for &v in &csp.neighbors[var]{if !assignment.contains_key(&v){for &x in &domains[v]{if val==x{n+=1;}}}} pairs.push((val,n));} pairs.sort_by_key(|&(_,n)|n); pairs.into_iter().map(|(v,_)|v).collect()
    }

    fn bt_improved(csp:&mut Csp,assignment:&mut BTreeMap<usize,u8>,domains:&mut Vec<Vec<u8>>)->bool {
        if csp.is_complete(assignment){return true;} let var=select_mrv(csp,assignment,domains);
        for val in order_lcv(csp,var,assignment,domains){csp.num_assignments+=1;if csp.is_consistent(var,val,assignment){assignment.insert(var,val);if bt_improved(csp,assignment,domains){return true;}assignment.remove(&var);csp.num_backtracks+=1;}} false
    }

    fn fc_propagate(csp:&Csp,var:usize,val:u8,assignment:&BTreeMap<usize,u8>,domains:&mut Vec<Vec<u8>>)->(Vec<(usize,u8)>,bool){
        let mut rem=Vec::new(); for &n in &csp.neighbors[var]{if !assignment.contains_key(&n){let mut del=Vec::new();for &x in &domains[n]{if val==x{del.push(x);rem.push((n,x));}}for x in del{domains[n].retain(|&y|y!=x);}if domains[n].is_empty(){return(rem,false);}}} (rem,true)
    }
    fn fc_restore(domains:&mut Vec<Vec<u8>>,rem:&[(usize,u8)]){for &(v,x) in rem{domains[v].push(x);}}
    fn fc_solve(csp:&mut Csp,assignment:&mut BTreeMap<usize,u8>,domains:&mut Vec<Vec<u8>>)->bool {
        if csp.is_complete(assignment){return true;} let var=select_mrv(csp,assignment,domains); for val in order_lcv(csp,var,assignment,domains){csp.num_assignments+=1;if csp.is_consistent(var,val,assignment){assignment.insert(var,val);let(rem,ok)=fc_propagate(csp,var,val,assignment,domains);if ok&&fc_solve(csp,assignment,domains){return true;}fc_restore(domains,&rem);assignment.remove(&var);csp.num_backtracks+=1;}} false
    }

    /// Contribution initiale : Evariste BALVAY (PR #378).
    struct ConflictBackjumping;
    impl ConflictBackjumping {
        fn select(csp:&Csp,assignment:&BTreeMap<usize,u8>)->usize{let mut best=None;for &(r,c) in &csp.variables{let v=r*9+c;if assignment.contains_key(&v){continue;}let n=csp.domains[v].iter().filter(|&&x|csp.is_consistent(v,x,assignment)).count();if best.map_or(true,|(m,_):(usize,usize)|n<m){best=Some((n,v));}}best.map(|(_,v)|v).unwrap_or(0)}
        fn solve_cbj(csp:&mut Csp,assignment:&mut BTreeMap<usize,u8>,sets:&mut BTreeMap<usize,HashSet<usize>>,history:&mut Vec<usize>)->(Option<BTreeMap<usize,u8>>,Option<usize>){
            if assignment.len()==csp.variables.len(){return(Some(assignment.clone()),None);}let var=Self::select(csp,assignment);history.push(var);sets.insert(var,HashSet::new());
            for val in csp.domains[var].clone(){let mut reject=HashSet::new();for &n in &csp.neighbors[var]{if let Some(&x)=assignment.get(&n){if val==x{reject.insert(n);}}}if !reject.is_empty(){sets.entry(var).or_default().extend(reject);continue;}assignment.insert(var,val);csp.num_assignments+=1;let(result,jump)=Self::solve_cbj(csp,assignment,sets,history);if result.is_some(){return(result,None);}assignment.remove(&var);if let Some(t)=jump{if t!=var{history.pop();return(None,Some(t));}}}
            csp.num_backtracks+=1;history.pop();let blames=sets.get(&var).cloned().unwrap_or_default();if blames.is_empty(){return(None,None);}let target=history.iter().rev().find(|&&v|blames.contains(&v)).copied();let target=match target{Some(t)=>t,None=>return(None,None)};let mut fusion=blames.clone();fusion.remove(&target);sets.entry(target).or_default().extend(fusion);(None,Some(target))
        }
        fn solve(csp:&mut Csp)->Option<BTreeMap<usize,u8>>{let mut a=BTreeMap::new();let mut s=BTreeMap::new();for &(r,c) in &csp.variables{s.insert(r*9+c,HashSet::new());}let mut h=Vec::new();Self::solve_cbj(csp,&mut a,&mut s,&mut h).0}
    }

    #[derive(Clone,Copy)] enum CspStrategyCbj{BacktrackingMrvLcv,ConflictBackJumping,ForwardChecking}
    impl CspStrategyCbj{fn nom(&self)->&'static str{match self{Self::BacktrackingMrvLcv=>"BACKTRACKING MRV LCV",Self::ConflictBackJumping=>"CONFLICT BACK JUMPING",Self::ForwardChecking=>"FORWARD CHECKING"}}}

    fn solve_with_strategy_cbj(grid:&SudokuGrid,strategy:CspStrategyCbj)->(Option<SudokuGrid>,u64,u64,f64){
        let mut csp=build_csp(grid);let mut assignment=BTreeMap::new();let start=Instant::now();let found=match strategy{
            CspStrategyCbj::ConflictBackJumping=>ConflictBackjumping::solve(&mut csp),
            CspStrategyCbj::BacktrackingMrvLcv=>{let mut d=csp.copy_domains();let ok=bt_improved(&mut csp,&mut assignment,&mut d);if ok{Some(assignment.clone())}else{None}},
            CspStrategyCbj::ForwardChecking=>{let mut d=csp.copy_domains();let ok=fc_solve(&mut csp,&mut assignment,&mut d);if ok{Some(assignment.clone())}else{None}},
        };let elapsed=start.elapsed().as_secs_f64();if let Some(a)=found{let mut r=grid.clone();appliquer(&mut r,&a);return(Some(r),csp.num_assignments,csp.num_backtracks,elapsed);}(None,csp.num_assignments,csp.num_backtracks,elapsed)
    }

    pub fn run(){
        let dir=std::env::current_dir().unwrap_or_else(|_|std::path::PathBuf::from("."));let puzzles=load_puzzles(&dir.join("Puzzles").join("Sudoku_Easy51.txt"),Some(5));
        // Benchmark sur 3 puzzles faciles (CBJ avec MRV converge, les 3 méthodes sont comparables)
        let strategies=[CspStrategyCbj::BacktrackingMrvLcv,CspStrategyCbj::ConflictBackJumping,CspStrategyCbj::ForwardChecking];let n=std::cmp::min(3,puzzles.len());
        println!("=== Benchmark CBJ vs heuristiques sur {} puzzles faciles ===",n);println!("{}","=".repeat(80));println!("{:<25} {:>10} {:>12} {:>12} {:>10}","Strategie","Assigns","Backtracks","Temps(ms)","Succes");println!("{}","-".repeat(80));
        for strategy in strategies{let mut ta=0u64;let mut tb=0u64;let mut tt=0.0;let mut ok=0;for i in 0..n{let grid=SudokuGrid::from_string(&puzzles[i]).unwrap_or_else(|_|SudokuGrid{cells:[[0u8;9];9]});let(r,a,b,t)=solve_with_strategy_cbj(&grid,strategy);ta+=a;tb+=b;tt+=t;if let Some(r)=r{if r.is_valid(){ok+=1;}}}println!("{:<25} {:>10} {:>12} {:>12} {}/{}",strategy.nom(),ta/n as u64,tb/n as u64,format!("{:.0}",tt*1000.0),ok,n);}println!("{}","=".repeat(80));
        // Lecture : CBJ sans heuristiques classiques (MRV seul ici) est competitif sur les
        // puzzles faciles, mais devient quadratique sur les puzzles difficiles à cause de
        // la sélection MRV statique. MAC et Forward Checking restent superieurs grâce à la
        // propagation. Le vrai gain du CBJ se manifeste sur les CSP non-binaires ou quand
        // les backtracks chronologiques explorent des branches eloignees du vrai conflit.
    }
}

strategies_cbj_38::run();

=== Benchmark CBJ vs heuristiques sur 3 puzzles faciles ===


Strategie                    Assigns   Backtracks    Temps(ms)     Succes


--------------------------------------------------------------------------------


BACKTRACKING MRV LCV             366           10           21 3/3


CONFLICT BACK JUMPING             82            1           14 3/3


FORWARD CHECKING                  91           10           16 3/3


### Lecture du benchmark : ce que les chiffres révèlent du « piège CBJ »

Le tableau ci-dessus confirme chiffrée la mise en garde énoncée plus haut, et révèle une nuance que la prose qualitative ne disait pas :

| Stratégie | Assignations | Backtracks | Temps (ms) — témoin Python |
|-----------|-------------|------------|------------|
| Backtracking MRV+LCV (nu) | **366** | 10 | 138 |
| Conflict-Based Backjumping | **82** | **1** | 91 |
| Forward Checking | 91 | 10 | 92 |

Les **assignations** et les **backtracks** sont des compteurs d'opérations : ils se lisent à l'identique dans la sortie de ce port. Les durées, elles, dépendent du matériel : la colonne cite les valeurs du témoin Python commité ; ce port Rust affiche ses propres mesures dans la sortie de la cellule précédente, et le rapport ligne à ligne est établi dans le corps de la PR.

**Deux observations honnêtes, l'une attendue, l'autre plus subtile.**

1. **Le backjumping tient sa promesse théorique — sur les backtracks.** CBJ divise les assignations par ~4,5 (366 → 82) et les retours arrière par 10 (10 → 1) face au backtracking MRV+LCV nu. En évitant de remonter chronologiquement vers des variables sans lien avec le conflit, CBJ court-circuite effectivement les branches mortes : c'est exactement ce que prédisait la théorie de Prosser (1993).

2. **Mais sur ces puzzles faciles, CBJ ≈ Forward Checking.** Regardez la dernière ligne : FC fait 91 assignations et 10 backtracks, quand CBJ fait 82 assignations et **1 seul** backtrack. Le backjumping divise pourtant les backtracks par 10 (10 → 1) sans se traduire en gain de temps mesurable : sur le témoin, l'écart entre CBJ (91 ms) et FC (92 ms) est de l'ordre de la milliseconde, noyé dans le bruit d'une exécution. Pourquoi ? Parce que la gestion des *conflict sets* (détection, fusion au backjump) a un **coût de bookkeeping** qui compense exactement le gain de retours arrière évités — et parce que sur ces puzzles faciles, la **propagation de contraintes de FC supprime déjà l'essentiel** des branches mortes avant même qu'un conflit ne se matérialise.

**C'est tout le sens du « piège » annoncé.** Sur des puzzles faciles, ce n'est pas le backjumping qui fait le travail, c'est la propagation : FC (propagation à 1 niveau) suffit, et CBJ n'apporte qu'un gain marginal (82 vs 91 assignations) par-dessus. La supériorité théorique de CBJ — remonter au coupable d'un conflit plutôt qu'au prédécesseur immédiat — ne paie vraiment que sur des puzzles **difficiles** comportant de **longues chaînes de conflit** que le backtracking chronologique explorerait vainement. C'est précisément pourquoi le framing prévenait qu'« il faut le combiner avec des heuristiques de sélection **et** de propagation de contraintes » : CBJ seul, sans FC/AC-3, plafonne là où la propagation serait déterminante.

> **À retenir.** Un benchmark sur des instances faciles peut faire croire qu'un algorithme « théoriquement supérieur » tient sa promesse (CBJ a bien 1 backtrack contre 10) tout en masquant qu'il n'apporte rien de mesurable par-dessus une propagation simple (CBJ ≈ FC en temps). La leçon « ne jugez pas un algorithme seulement sur des instances faciles » se lit ici dans les chiffres : pour départager CBJ et FC, il faudrait des puzzles où la profondeur des conflits fait décoller CBJ — ou, à l'inverse, où l'overhead de bookkeeping le fait perdre.

## Exercice : Comparaison Forward Checking vs MAC sur le Coloriage de Graphe

### Enonce

Le **coloriage de graphe** est un CSP classique : attribuer une couleur à chaque sommet d'un graphe tel que deux sommets adjacents n'aient jamais la même couleur. C'est un problème fondamental en IA avec des applications en planification, allocation de ressources et registres CPU.

Contrairement au Sudoku (grille structuree), le coloriage de graphe offre des topologies variées qui mettent en evidence les différences entre Forward Checking et MAC de manière plus ou moins prononcée selon la densite du graphe.

### Objectif

Implémentez la fonction `solve_graph_coloring` qui resout un problème de coloriage de graphe en utilisant la structure `Csp` générique définie plus haut. Comparez ensuite les performances de Forward Checking et MAC sur des graphes de densite variable.

> **Étapes :**

1. **Construire** le CSP du coloriage de graphe :
   - Variables : les sommets du graphe (entiers 0 à n-1)
   - Domaines : `{0, 1, ..., k-1}` pour k couleurs
   - Contraintes : sommets adjacents doivent avoir des couleurs différentes
2. **Résoudre** avec Forward Checking et MAC
3. **Mesurer** assignations, backtracks et temps pour chaque méthode
4. **Afficher** le résultat sous forme de tableau comparatif

> **Indices :**

- Utilisez la structure `Csp` existante avec la contrainte binaire `val1 != val2`
- La fonction `generate_random_graph(n, edge_prob)` génère un graphe aléatoire (modèle Erdos-Renyi)
- Un graphe avec `edge_prob=0.3` et 15 sommets est un bon point de départ pour 3 couleurs

In [20]:
// Stub C.1 : coloriage de graphe
mod graph_stub_41 {
    use std::collections::BTreeMap;

    /// CSP generique binaire : variables = indices de sommets du graphe.
    struct Csp {
        variables: Vec<usize>,
        domains: Vec<Vec<u8>>,
        neighbors: Vec<Vec<usize>>,
        num_assignments: u64,
        num_backtracks: u64,
    }

    impl Csp {
        fn new() -> Self {
            Csp {
                variables: Vec::new(),
                domains: Vec::new(),
                neighbors: Vec::new(),
                num_assignments: 0,
                num_backtracks: 0,
            }
        }

        fn get_arcs(&self) -> Vec<(usize, usize)> {
            let mut arcs = Vec::new();
            for var in 0..self.variables.len() {
                for &voisin in &self.neighbors[var] {
                    arcs.push((var, voisin));
                }
            }
            arcs
        }
    }

    /// Genere un graphe aleatoire (modele Erdos-Renyi).
    ///
    /// n : Nombre de sommets
    /// edge_prob : Probabilite d'avoir une arete entre deux sommets
    /// seed : Graine aleatoire pour la reproductibilite
    ///
    /// Rend le dictionnaire {sommet: [voisins]}
    fn generate_random_graph(_n: usize, _edge_prob: f64, _seed: u64) -> BTreeMap<usize, Vec<usize>> {
        // TODO : Implementer la generation du graphe
        // Pour chaque paire (i, j) avec i < j :
        //   - Tirer un nombre aleatoire
        //   - Si random < edge_prob, ajouter l'arete (i <-> j)
        // Note : le PRNG devra etre nomme et fixe (constante de graine) ;
        // ne jamais comparer des nombres tires a ceux du jumeau Python.
        BTreeMap::new()
    }

    /// Construit le CSP du coloriage de graphe.
    ///
    /// adjacency : Dictionnaire {sommet: [voisins]}
    /// num_colors : Nombre de couleurs disponibles
    ///
    /// Rend l'instance CSP pour le coloriage
    fn build_graph_coloring_csp(_adjacency: &BTreeMap<usize, Vec<usize>>, _num_colors: usize) -> Csp {
        // TODO : Construire le CSP
        // 1. variables = liste des sommets
        // 2. domains = {sommet: list(range(num_colors))} pour chaque sommet
        // 3. neighbors = adjacency (déjà le bon format)
        // 4. contrainte binaire : val1 != val2
        Csp::new()
    }

    /// Resout un probleme de coloriage de graphe et compare FC vs MAC.
    ///
    /// n : Nombre de sommets
    /// edge_prob : Densite des aretes
    /// num_colors : Nombre de couleurs
    fn solve_graph_coloring(_n: usize, _edge_prob: f64, _num_colors: usize) {
        // TODO : Implementer la résolution complète
        // 1. Generer le graphe avec generate_random_graph
        // 2. Construire le CSP avec build_graph_coloring_csp
        // 3. Résoudre avec fc_solve et mac_solve
        // 4. Afficher les résultats sous forme de tableau :
        //    Méthode | Assignations | Backtracks | Temps(ms) | Succes
    }

    pub fn run() {
        // Test
        println!("=== Coloriage de graphe : Forward Checking vs MAC ===");
        println!();

        // Graphe peu dense (FC et MAC similaires)
        println!("--- Graphe peu dense (edge_prob=0.2, 15 sommets, 3 couleurs) ---");
        solve_graph_coloring(15, 0.2, 3);

        println!();

        // Graphe plus dense (MAC devrait montrer un avantage)
        println!("--- Graphe dense (edge_prob=0.4, 15 sommets, 4 couleurs) ---");
        solve_graph_coloring(15, 0.4, 4);
    }
}

graph_stub_41::run();

=== Coloriage de graphe : Forward Checking vs MAC ===


--- Graphe peu dense (edge_prob=0.2, 15 sommets, 3 couleurs) ---


--- Graphe dense (edge_prob=0.4, 15 sommets, 4 couleurs) ---


### Pourquoi la sortie n'affiche que des en-têtes

Les deux blocs d'expérimentation (graphe peu dense, graphe dense) impriment leur en-tête, puis **aucune ligne de résultat** : c'est l'état attendu d'un stub. Le corps neutre de `generate_random_graph` ne produit pas de graphe, donc `solve_graph_coloring` n'a rien à comparer — la structure du benchmark (l'appel aux deux solveurs et le tableau attendu) est en place, mais chaque ligne du tableau dépend du générateur que l'exercice demande d'implémenter. C'est volontairement que la comparaison FC vs MAC vous appartient : mesurez assignations et backtracks sur les deux densités, et confrontez ce que vous obtenez à ce que la section 8 a montré sur le Sudoku.

## Synthèse et perspectives

Ce notebook a formalisé le Sudoku comme un problème de satisfaction de contraintes (CSP) binaire selon le cadre académique AIMA (Russell & Norvig, Chapitre 6), avec 81 variables, des domaines à 9 valeurs et 27 contraintes AllDifferent decomposees en paires. Quatre algorithmes de résolution ont été implémentés et comparés : le backtracking simple (2,9 millions d'assignations), le backtracking améliore avec MRV et LCV (255 assignations), le Forward Checking (81 assignations) et le MAC (81 assignations, 0 backtrack). L'écart de quatre ordres de grandeur entre le backtracking naïf et les méthodes avec propagation illustré l'importance cruciale de la reduction des domaines.

L'étude du Conflict-Based Backjumping (CBJ) a fourni un exemple cautionnaire instructif : bien que théoriquement superieur au backtracking chronologique, CBJ sans propagation de contraintes ne surpasse pas Forward Checking ou MAC sur les puzzles difficiles. Cette observation rappelle que le choix d'un algorithme ne se juge pas sur les instances faciles seul. L'exercice sur le coloriage de graphe permet de transferer ces compétences vers un autre CSP classique avec des topologies variees.

Le notebook suivant, [Sudoku-07-Norvig-Rust](Sudoku-07-Norvig-Rust.ipynb), présente l'approche de Peter Norvig, qui combine la propagation de contraintes (elimination et unique candidat restant) avec une recherche en profondeur pour résoudre efficacement tout Sudoku.

## Synthèse

### Algorithmes implémentés

| Algorithme | Propagation | Détection d'échec | Performance |
|------------|-------------|-------------------|-------------|
| **Backtracking** | Aucune | A l'assignation | Lente |
| **BT + MRV + LCV** | Aucune | A l'assignation | Amelioree |
| **Forward Checking** | 1 niveau | Domaine voisin vide | Rapide |
| **MAC** | Complète (AC-3) | Domaine vide global | Optimale |

### Heuristiques

| Heuristique | Rôle | Effet |
|-------------|------|-------|
| **MRV** | Sélection de variable | Fail-first |
| **LCV** | Ordonnancement valeurs | Succeed-first |
| **Degree** | Departage MRV | Priorité aux variables contraintes |

### Liens avec les autres notebooks

- **Sudoku-08-HumanStrategies** : Propagation plus poussées (naked/hidden singles)
- **Sudoku-10-ORTools** : Bibliothèque industrielle avec propagation optimisee
- **Sudoku-12-Z3** : SMT solver

### References

- Russell, S. & Norvig, P. *Artificial Intelligence: A Modern Approach*, 4e ed., Chapitre 6
- Mackworth, A. K. *Consistency in Networks of Relations* (1977)
- Dechter, R. *Constraint Processing*, Cambridge University Press, 2003

***

**Navigation** : [<< Sudoku-05-PSO](Sudoku-05-PSO-Rust.ipynb) | [Index](README.md) | [Sudoku-07-Norvig >>](Sudoku-07-Norvig-Rust.ipynb)

## À retenir

La formulation **CSP** (Constraint Satisfaction Problem) du Sudoku, telle que présentée dans *AIMA* (Russell & Norvig, ch. 6), est la plus puissante de la série : **81 variables** (les cases), **9 valeurs** par domaine, et des contraintes de différence. Les algorithmes de CSP y sont nettement plus efficaces que le backtracking brut.

- Le test **MAC** (Maintaining Arc Consistency) résout la grille témoin avec **81 assignations et 0 backtracks** — l'arc-consistance élimine les valeurs impossibles avant même de brancher. La durée affichée par ce port est mesurée à l'exécution du carnet : elle se lit en ordre de grandeur, seule la paire 81/0 est stable.
- Sur 2 puzzles faciles, le backtracking simple effectue **2 889 322 assignations**, contre **255** pour MRV+LCV et **81** pour MAC : un écart de **quatre ordres de grandeur**.
- Sur les puzzles durs (top-95), **Forward Checking** (2143 assignations / 2062 backtracks) et **MAC** (1060 assignations / 979 backtracks) restent du même ordre, là où le brut exploserait.

## Vérifiez votre compréhension

**1. Pourquoi MAC n'a-t-il besoin que de 81 assignations et 0 backtracks ?**

<details><summary>Réponse</summary>

**MAC** (Maintaining Arc Consistency) propage les contraintes *à chaque pas* : dès qu'une case est assignée, les valeurs rendues incompatibles sont retirées des domaines des cases voisines, et la propagation se propage. Quand l'algorithme vient à brancher, le domaine est déjà tant réduit qu'aucun faux pas n'est nécessaire — d'où **81 assignations** (une par case) et **0 backtracks** sur la grille témoin, en quelques millisecondes mesurées par ce port.

</details>

**2. Que mesure le benchmark sur les 2 puzzles faciles, et que prouve-t-il ?**

<details><summary>Réponse</summary>

Sur les 2 puzzles faciles : **BACKTRACKING SIMPLE 2 889 322** assignations, **MRV+LCV 255**, **FORWARD CHECKING 81**, **MAC 81**. L'écart de **~2.9 millions à 81** illustre l'effet cumulé des heuristiques d'ordonnancement (MRV/LCV) et de la propagation d'arc : même grille, même problème, mais un coût qui dépend de *l'algorithme*, pas du puzzle.

</details>

**3. Sur les puzzles durs (top-95), Forward Checking et MAC sont-ils au coude-à-coude ?**

<details><summary>Réponse</summary>

Oui : sur les 4 puzzles du top-95, **Forward Checking compte 2143 assignations et 2062 backtracks** et **MAC 1060 assignations et 979 backtracks** — même ordre de grandeur. Sur les instances les plus dures, l'avantage de MAC se resserre (la propagation coûte), mais les deux restent infimes face au backtracking brut. C'est la limite classique : sur les instances les plus dures, les méthodes de propagation convergent.

</details>

## Pour aller plus loin

- **[Sudoku-07 — Norvig (Propagation)](Sudoku-07-Norvig-Rust.ipynb)** : le carnet suivant — l'implémentation de Peter Norvig, la même propagation de contraintes dans un style épuré.

- [Sudoku-06 — CSP (Python)](Sudoku-06-AIMA-CSP-Python.ipynb) : le jumeau Python de la formulation CSP.

- [Sudoku-06 — CSP (C#)](Sudoku-06-AIMA-CSP-CSharp.ipynb) : le jumeau C# de la formulation CSP.

### Lectures

- Russell & Norvig, *Artificial Intelligence: A Modern Approach*, ch. 6 — les CSP, MRV/LCV, forward checking et arc consistency.
